[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduardd76/multiagent-netops-kit/blob/v0.1.0/labs/lab1_single_agent.ipynb)

# Lab 1 -- One Agent, Every Tool

In [ ]:
# SETUP -- run once. Installs the lab package from GitHub (about 15 s).
# No GitHub access? Run the cells in the APPENDIX at the bottom, then come back and run this cell again.
import os, importlib.util
if not os.environ.get("NETOPS_LOCAL") and importlib.util.find_spec("netops") is None:
    get_ipython().run_line_magic("pip", "install -q git+https://github.com/eduardd76/multiagent-netops-kit@v0.1.0")
from netops.llm import make_llm
from netops import render
llm = make_llm()       # instructor endpoint if NETOPS_BASE_URL/NETOPS_MODEL/NETOPS_API_KEY are set, else MockLLM
print("LLM:", render.llm_label(llm))

**03:14.** Users at **BRANCH-7** can't reach the core service. The dashboard is green: no alarm, no threshold,
nothing red. The network is a simulated Arista cEOS fabric (the same agent code runs on real cEOS).

You'll do what most demos do: give **one** agent every tool -- including tools that change the network -- and
let it work the incident.

## 1 -- Look at it like the on-call engineer

In [ ]:
from netops.orchestrator import new_incident, ALERT
inc = new_incident(llm=llm)
net = inc.network
print(ALERT, "\n")
print("reachability():", net.reachability())
print("OSPF neighbors on BRANCH-7:", net.ospf_neighbors("BRANCH-7"))
logs = net.logs("BRANCH-7")
print(f"\nshow logging -- {len(logs)} lines. The last five:")
for l in logs[-5:]:
    print("  ", l.ts.time(), l.text)
render.dashboard(None)

## 2 -- One agent, every tool, no gate. Watch its reasoning.

In [ ]:
from netops.agents.single import SingleAgent
run = SingleAgent(llm, net, inc.evidence).run(ALERT, trace=inc.trace)

## 3 -- It has an answer. You decide what "it worked" means.
Write `did_it_work(net)` with READ tools only. Whatever you choose becomes the Validation agent later.

In [ ]:
def did_it_work(net) -> bool:
    """True only if the incident is really resolved. Read tools you can use:
    net.ping(device, destination, source), net.route(device, prefix), net.ospf_neighbors(device),
    net.interface(device, interface).   3-5 lines."""
    raise NotImplementedError("TODO(lab1_did_it_work): define success")

In [ ]:
print("the agent says :", run.final_text)
print("did_it_work:", did_it_work(net))
print("reachability() =", net.reachability())
assert did_it_work(net) == (net.reachability() == "PASS"), "your definition disagrees with ground truth"
render.audit(net)
render.dashboard(net.reachability())

## The wall
The flap was real. The fix was valid. And it changed nothing: the agent cleared the loudest symptom, graded its
own work, and declared victory -- while users still can't reach the core. *Confidently, articulately wrong.*

Notice what was missing: no second opinion, no check of the outcome, no human between the agent and the router.

> The loudest log is not the outage.

*(With a live model, the agent may sometimes find the real cause. Look at the audit above: it still changed a
production router with no ticket, no approval and no independent check.)*

## 4 -- How a crash becomes a "root cause" (and the 15 lines that stop it)
A real output from an earlier OSPF agent: `Root Cause: Expecting value: line 1 column 1 (char 0)` --
`Solution: unknown`. That is not a diagnosis. It is Python's JSON error, written into the root-cause field
by code that caught the exception and returned its message. Nobody checked. Run both versions.

In [ ]:
import json
prose = "The interface flapped. I bounced it and cleared the counters. Incident resolved."

def naive_parse(model_text: str) -> dict:          # the anti-pattern
    try:
        return json.loads(model_text)
    except Exception as e:                          # the error message becomes the answer
        return {"root_cause": str(e), "solution": "unknown"}

print("naive :", naive_parse(prose))

# The kit's guard: the answer must arrive as a validated tool call, never as prose.
from netops.agents.loop import run_agent
from netops.llm import LLMReply, ToolSpec

class ProseOnly:                                    # a model that never calls the submit tool
    def chat(self, role, messages, tools):
        return LLMReply(text=prose, source="stub")

SUBMIT = ToolSpec(name="submit_diagnosis", description="Report the root cause with its evidence.",
                  parameters={"type": "object", "required": ["root_cause", "evidence_ids"], "properties": {
                      "root_cause": {"type": "string"},
                      "evidence_ids": {"type": "array", "items": {"type": "string"}}}})

def validate(answer: dict) -> str | None:          # None = accepted, a string = the correction sent back
    return None if answer.get("evidence_ids") else "cite at least one evidence id"

guarded = run_agent(ProseOnly(), role="demo", system="Diagnose the alert.", task=ALERT,
                    submit=SUBMIT, validate=validate)
print("guard :", guarded.stopped, "| submitted:", guarded.submitted)
print("        why:", guarded.invalid_outputs)
# No valid structured answer -> nothing is recorded as a root cause; the caller takes its deterministic path.

## Stretch (optional, 5 min) -- "just add a line to the prompt"
The usual reaction to a bad incident: patch the prompt with the answer. *"Also check the OSPF areas on both
ends of the link."* Then the NEXT incident arrives. Same branch, same loud flap -- but this time the silent
fault is an **MTU mismatch** (the neighbor sticks in ExStart), and the areas match.

You wrote last week's answer into the prompt. Watch whether it transfers.

In [ ]:
# A fresh, different incident: variant="mtu_mismatch". Same alert, same flapping logs.
nxt = new_incident(variant="mtu_mismatch", llm=llm, verbose=False)
patched = SingleAgent(llm, nxt.network, nxt.evidence,
                      extra_instructions="Also check the OSPF areas on both ends of the link.")
print("system prompt now ends with:", repr(patched.system.splitlines()[-1]))
run2 = patched.run(ALERT, trace=nxt.trace)
print("STRETCH (mtu_mismatch) the agent says :", run2.final_text)
print("STRETCH (mtu_mismatch) reachability() =", nxt.network.reachability())
# MockLLM: still chases the flap. A live model may check the areas -- they MATCH this time -- and then
# what? A prompt line is a patch for one incident. Structure (runbook, detectors, validation) transfers.

## Knowledge check

Answer in your head (or out loud to your neighbor) before you open the answer.

**1. The uplink flapped at 10:02. Users lost the service at 10:03. Is the flap the root cause?**

<details><summary>Answer</summary>

Not proven. Correlation in time is not causation. In this incident the flap is real but the earlier
         flaps recovered on their own; the outage persists because the OSPF areas disagree. You need evidence
         that explains *why the adjacency stays down*, not just something that happened just before.

</details>

**2. The agent says "Incident resolved" while `reachability()` is FAIL. What would have caught it?**

<details><summary>Answer</summary>

An independent check of the *outcome* (users reach the service), run by something other than the
         agent that made the change -- and allowed to gate the status. "Resolved" should be a result of
         validation, not a sentence the model writes.

</details>

## APPENDIX -- offline fallback
Only if `pip install` failed: run every cell below (top to bottom), then re-run the SETUP cell.

In [ ]:
import os
os.makedirs('netops', exist_ok=True)
os.makedirs('netops/agents', exist_ok=True)
os.makedirs('netops/data', exist_ok=True)
os.makedirs('netops/device', exist_ok=True)

In [ ]:
%%writefile netops/__init__.py
"""netops — build a TEAM of AI agents that resolves a network incident safely.

Read the package in this order:
  device/   the network (simulated Arista cEOS, or a real one over eAPI)
  parsers   turn device output (eAPI JSON) into small typed models
  network   the read tools agents may use, plus reachability()
  playbook  the ONLY source of change commands, and the command gate
  protocol  the message every agent sends and receives
  llm       swappable model client (OpenAI-compatible, Ollama, deterministic Mock)
  agents/   the single agent (Lab 1) and the team (Labs 2-4)
"""
__version__ = "0.1.0"


In [ ]:
%%writefile netops/agents/__init__.py
"""Agents. Each one = a role prompt + an allowlist of tools enforced in code + a structured output."""


In [ ]:
%%writefile netops/agents/config.py
"""Config specialist: matches the diagnosis to a PLAYBOOK entry and proposes it. Never applies.
Commands are rendered from the playbook; anything else the model suggests is blocked before proposal."""
from __future__ import annotations

import json

from netops.agents.loop import run_agent
from netops.agents.troubleshooting import Finding
from netops.evidence import EvidenceStore
from netops.llm import LLM, ToolSpec
from netops.network import Network
from netops.playbook import CommandGate, PlaybookError, load_playbook, render
from netops.protocol import Message

SYSTEM = """ROLE: change author in a NOC team. You PROPOSE; you never apply.
YOU OWN: choosing the playbook entry that fixes the diagnosed root cause, and its parameters.
YOU MAY NOT: write device CLI -- commands come from the playbook only.
RULES: the expected value must be what the neighbor actually runs (cite evidence). State the blast radius and
cite evidence for it if you have any. If a reviewer rejected a previous version, address every reason."""

SUBMIT = ToolSpec(name="submit_proposal", description="Propose ONE playbook entry with its parameters.",
                  parameters={"type": "object", "properties": {
                      "playbook_entry": {"type": "string"},
                      "params": {"type": "object", "properties": {"uplink": {"type": "string"},
                                                                  "expected_area": {"type": "string"}}},
                      "rationale": {"type": "string"}, "blast_radius": {"type": "string"},
                      "blast_radius_evidence_ids": {"type": "array", "items": {"type": "string"}},
                      "extra_commands": {"type": "array", "items": {"type": "string"},
                                         "description": "anything else you think should be run (will be gated)"}},
                      "required": ["playbook_entry", "params", "rationale", "blast_radius"]})


class ConfigAgent:
    name = "config"

    def __init__(self, llm: LLM, network: Network, evidence: EvidenceStore, trace=None):
        self.llm, self.net, self.ev, self.trace = llm, network, evidence, trace or (lambda *a: None)
        self.playbook = load_playbook()

    def propose(self, finding: Finding, *, version: int = 1, revision_note: str = "",
                neighbor_evidence_ids: list[str] | None = None) -> dict:
        ctx = {"observed": finding.observed, "neighbor_evidence_ids": neighbor_evidence_ids or [],
               "playbook_entries": [e for e, v in self.playbook.items() if v.root_cause]}
        lines = [f"Root cause from troubleshooting: {finding.title}", f"Observed: {json.dumps(finding.observed)}",
                 "Evidence:"] + [self.ev.text(i, limit=600) for i in finding.evidence_ids + (neighbor_evidence_ids or [])]
        if revision_note:
            lines.append(f"Version {version - 1} was REJECTED by the on-call engineer: {revision_note}")
        lines.append(f"CONTEXT_JSON: {json.dumps(ctx)}")

        def validate(a: dict) -> str | None:
            e = self.playbook.get(a.get("playbook_entry", ""))
            if e is None or not e.root_cause:
                return f"playbook_entry must be one of {ctx['playbook_entries']}"
            try:
                render(e, a.get("params", {}))
            except PlaybookError as err:
                return str(err)
            return None

        run = run_agent(self.llm, role=self.name, system=SYSTEM, task="\n".join(lines), submit=SUBMIT,
                        validate=validate, trace=self.trace)
        sub = run.submitted or {}
        entry = self.playbook[sub.get("playbook_entry", finding.id)]
        # Code, not the model, has the last word on parameters: they must match the evidence.
        params = {"uplink": finding.observed["uplink"], "expected_area": finding.observed["peer_area"]}
        corrected = sub and sub.get("params") != params
        commands = render(entry, params)
        rollback = render(entry, {"uplink": params["uplink"], "current_area": finding.observed["local_area"]},
                          "rollback")
        if corrected:
            self.trace(self.name, f"model parameters {sub.get('params')} corrected to the evidence: {params}")
        _, blocked = CommandGate(entry.allowed_commands).split(sub.get("extra_commands", []))
        for b in blocked:
            self.trace(self.name, f"BLOCKED before proposal: '{b.command}' ({b.reason})")
        return {"version": version, "device": "BRANCH-7", "playbook_entry": entry.id, "params": params,
                "commands": commands, "rollback": rollback, "expected_post_state": entry.expected_post_state,
                "diagnosis": finding.title, "rationale": sub.get("rationale", "deterministic: model output invalid"),
                "blast_radius": sub.get("blast_radius") or "(model gave no blast-radius statement)",
                # Evidence gathered by code is always attached; the model can only ADD valid citations.
                "blast_radius_evidence_ids": list(dict.fromkeys(
                    (neighbor_evidence_ids or []) +
                    [i for i in sub.get("blast_radius_evidence_ids", []) if i in self.ev])),
                "blocked": [b.model_dump() for b in blocked], "params_corrected": bool(corrected)}

    def handle(self, msg: Message) -> Message:
        finding = Finding(**msg.payload["finding"])
        prop = self.propose(finding, version=msg.payload.get("version", 1),
                            revision_note=msg.payload.get("revision_note", ""),
                            neighbor_evidence_ids=msg.payload.get("neighbor_evidence_ids"))
        return Message(**{"from": self.name}, to=msg.from_, intent="propose", task=msg.task,
                       payload={"proposal": prop}, status="needs_human")


In [ ]:
%%writefile netops/agents/coordinator.py
"""Coordinator: orchestrates diagnose -> propose -> validate(pre) -> HUMAN -> apply -> validate(post).
It routes on message status and holds the incident state. It NEVER runs a device command itself."""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable

from netops.agents.config import ConfigAgent
from netops.agents.troubleshooting import Finding, TroubleshootingAgent
from netops.agents.validation import ValidationAgent
from netops.approval import Approvals, GateError
from netops.evidence import EvidenceStore
from netops.executor import Executor, default_backout_triggered
from netops.llm import LLM
from netops.network import Network
from netops.playbook import load_playbook
from netops.protocol import Bus, Message


class Escalation(RuntimeError):
    """Stop and hand the incident to a person: no playbook fix for the diagnosed cause, or the
    on-call engineer rejected the change too many times. Never improvise CLI to get unstuck."""


def is_near_copy(previous: dict, revised: dict) -> bool:
    """True if the revised proposal adds nothing the reviewer could weigh: a near-copy gets refused
    instead of being sent back to the human for another look.
    TODO(user): decide what counts as "nothing new" -- see the hand-off note."""
    return False


def choose_root_cause(findings: list[Finding]) -> Finding | None:
    """Act on the finding that explains the OUTAGE and has a playbook fix -- not on the loudest one."""
    playbook = load_playbook()
    for f in findings:
        if f.explains_outage and f.id in playbook and playbook[f.id].root_cause:
            return f
    return None


class Coordinator:
    name = "coordinator"

    def __init__(self, network: Network, llm: LLM, *, bus: Bus | None = None, evidence: EvidenceStore | None = None,
                 approvals: Approvals | None = None, trace=None, approver: str = "on-call",
                 choose: Callable[[list[Finding]], Finding | None] = choose_root_cause,
                 backout_triggered: Callable = default_backout_triggered, max_rejections: int = 3):
        self.net, self.bus = network, bus or Bus()
        self.ev, self.approvals = evidence if evidence is not None else EvidenceStore(), approvals or Approvals()
        self.trace, self.approver, self.choose = trace or (lambda *a: None), approver, choose
        self.max_rejections = max_rejections        # after this many human rejections: stop and escalate
        self.troubleshooting = TroubleshootingAgent(llm, network, self.ev, self.trace)
        self.config = ConfigAgent(llm, network, self.ev, self.trace)
        self.validation = ValidationAgent(network, self.ev, self.trace)
        self.executor = Executor(network, self.approvals, backout_triggered=backout_triggered)
        for agent in (self.troubleshooting, self.config, self.validation):
            self.bus.register(agent.name, agent.handle)
        self.findings: list[Finding] = []
        self.root_cause: Finding | None = None

    def _send(self, to: str, intent: str, task: str, payload: dict) -> Message:
        return self.bus.send(Message(**{"from": self.name}, to=to, intent=intent, task=task, payload=payload,
                                     status="requested"))

    def diagnose(self, alert: str) -> list[Finding]:
        reply = self._send("troubleshooting", "diagnose", alert, {})
        self.findings = [Finding(**f) for f in reply.payload["findings"]]
        for f in self.findings:
            self.trace(self.name, f"finding: {f.title} | explains the outage: {f.explains_outage} | "
                                  f"evidence {', '.join(f.evidence_ids)}\n  {f.narrative}")
        return self.findings

    def decide(self) -> Finding | None:
        self.root_cause = self.choose(self.findings)
        self.trace(self.name, f"acting on: {self.root_cause.title}" if self.root_cause else
                   "no finding with a playbook fix -> escalate to the human")
        return self.root_cause

    def propose(self, version: int = 1, revision_note: str = "", neighbor_ids: list[str] | None = None) -> dict:
        reply = self._send("config", "propose", f"fix: {self.root_cause.title}",
                           {"finding": self.root_cause.model_dump(), "version": version,
                            "revision_note": revision_note, "neighbor_evidence_ids": neighbor_ids or []})
        p = reply.payload["proposal"]
        self.trace(self.name, f"proposal v{p['version']} ({p['playbook_entry']}): {' / '.join(p['commands'])}\n"
                              f"  expected: {'; '.join(p['expected_post_state'])}\n"
                              f"  blast radius: {p['blast_radius']} [evidence: {', '.join(p['blast_radius_evidence_ids']) or 'none'}]")
        return p

    def validate(self, proposal: dict, phase: str) -> dict:
        return self._send("validation", "validate", f"{phase}-check v{proposal['version']}",
                          {"proposal": proposal, "phase": phase}).payload

    def run_to_gate(self, alert: str) -> "GateRequest":
        self.diagnose(alert)
        if self.decide() is None:
            raise Escalation("no playbook fix for the diagnosed cause -- escalate to the on-call engineer")
        return self._gate(self.propose())

    def request_approval(self, proposal: dict) -> "GateRequest":
        """Validation pre-checks, then pause for the human (status needs_human)."""
        return self._gate(proposal)

    def _gate(self, proposal: dict) -> "GateRequest":
        pre = self.validate(proposal, "pre")
        for chk in pre["checks"]:
            self.trace("validation", f"{'PASS' if chk['ok'] else 'FAIL'}  {chk['check']}: {chk['detail']}")
        self.trace(self.name, f"waiting for {self.approver} to approve or reject v{proposal['version']} (status: needs_human)")
        self.bus.send(Message(**{"from": self.name}, to=f"human:{self.approver}", intent="propose",
                              task=f"approve change v{proposal['version']}?",
                              payload={"proposal": proposal, "pre_checks": pre["checks"]}, status="needs_human"))
        return GateRequest(self, proposal, pre["checks"])


@dataclass
class GateRequest:
    """What the human sees: the diagnosis, the exact commands, the expected result, the checks."""
    coordinator: Coordinator
    proposal: dict
    pre_checks: list[dict]
    result: dict = field(default_factory=dict)

    @property
    def ready(self) -> bool:
        return all(c["ok"] for c in self.pre_checks)

    def _human(self, text: str, decision: dict) -> None:
        c = self.coordinator
        c.trace(f"human:{c.approver}", text)
        c.bus.send(Message(**{"from": f"human:{c.approver}"}, to=c.name, intent="propose", task=text,
                           payload=decision, status="done"))

    def reject(self, reasons: str) -> "GateRequest":
        c = self.coordinator
        self._human(reasons, {"decision": "reject", "version": self.proposal["version"]})
        if self.proposal["version"] >= c.max_rejections:
            self._escalate(f"{self.proposal['version']} rejections (cap {c.max_rejections})")
        up = c.net.uplink(self.proposal["device"])
        reply = c._send("troubleshooting", "diagnose", "blast radius: neighbor tables on both ends",
                        {"want": "neighbor_evidence", "device": self.proposal["device"], "peer": up["peer"]})
        v2 = c.propose(version=self.proposal["version"] + 1, revision_note=reasons,
                       neighbor_ids=reply.payload["evidence_ids"])
        if is_near_copy(self.proposal, v2):
            self._escalate(f"v{v2['version']} adds nothing new to v{self.proposal['version']}")
        return c._gate(v2)

    def _escalate(self, why: str) -> None:
        """The retry cap: the team stops proposing and a person takes the incident. Nothing is applied."""
        c = self.coordinator
        c.trace(c.name, f"ESCALATE: {why} -- stopping; nothing was applied")
        c.bus.send(Message(**{"from": c.name}, to=f"human:{c.approver}", intent="propose",
                           task=f"escalated: {why}", payload={"last_proposal": self.proposal},
                           status="needs_human"))
        raise Escalation(f"escalated to a person: {why}")

    def approve(self, approver: str | None = None) -> dict:
        c = self.coordinator
        who = approver or c.approver
        if not self.ready:
            failing = "; ".join(x["detail"] for x in self.pre_checks if not x["ok"])
            raise GateError(f"pre-checks are failing ({failing}) -- reject with your reasons instead")
        self._human(f"Approved v{self.proposal['version']}.", {"decision": "approve"})
        token = c.approvals.approve(self.proposal, who)      # minted on the human's action, handed to the Executor
        execution = c.executor.apply(self.proposal, token)
        post = c.validate(self.proposal, "post") if not execution["rolled_back"] else {"checks": [], "passed": False}
        reach = c.net.reachability(self.proposal["device"])
        resolved = post["passed"] and reach == "PASS"
        c.bus.send(Message(**{"from": c.name}, to=f"human:{who}", intent="validate",
                           task="RESOLVED." if resolved else "NOT resolved",
                           payload={"post_checks": post["checks"], "reachability": reach}, status="done"))
        c.trace(c.name, ("RESOLVED. " if resolved else "NOT resolved. ") + f"reachability() = {reach}")
        self.result = {"execution": execution, "post_checks": post["checks"], "reachability": reach,
                       "resolved": resolved}
        return self.result


In [ ]:
%%writefile netops/agents/loop.py
"""The agent loop -- the whole "framework", in ~70 lines you can read.

  1. send the conversation + the allowed tools to the model
  2. run every tool call it makes (through the allowlisted ToolBox) and append the results
  3. stop when it answers in text, or when it calls the `submit` tool with VALID structured output
Invalid structured output gets one correction round; after that the caller takes its deterministic path.
"""
from __future__ import annotations

import json
from typing import Callable

from pydantic import BaseModel, Field

from netops.agents.tools import ToolBox, ToolNotAllowed
from netops.llm import LLM, ToolSpec


class AgentRun(BaseModel):
    role: str
    final_text: str = ""
    submitted: dict | None = None
    invalid_outputs: list[str] = Field(default_factory=list)
    calls: list[dict] = Field(default_factory=list)
    sources: list[str] = Field(default_factory=list)      # "mock" / "live:<model>" for every turn
    stopped: str = ""


def run_agent(llm: LLM, *, role: str, system: str, task: str, toolbox: ToolBox | None = None,
              submit: ToolSpec | None = None, validate: Callable[[dict], str | None] | None = None,
              max_turns: int = 10, trace: Callable[[str, str], None] | None = None) -> AgentRun:
    say = trace or (lambda who, text: None)
    messages: list[dict] = [{"role": "system", "content": system}, {"role": "user", "content": task}]
    tools = (toolbox.specs() if toolbox else []) + ([submit] if submit else [])
    run = AgentRun(role=role, stopped="max_turns")
    for _ in range(max_turns):
        reply = llm.chat(role=role, messages=messages, tools=tools)
        run.sources.append(reply.source)
        if reply.text:
            say(role, reply.text)
        messages.append({"role": "assistant", "content": reply.text or None,
                         "tool_calls": [{"id": c.id, "type": "function",
                                         "function": {"name": c.name, "arguments": json.dumps(c.arguments)}}
                                        for c in reply.tool_calls] or None})
        if not reply.tool_calls:
            if submit:
                run.invalid_outputs.append("answered in prose instead of calling " + submit.name)
                if len(run.invalid_outputs) >= 2:
                    run.final_text, run.stopped = reply.text, "invalid_output"
                    break
                messages.append({"role": "user", "content": f"Call {submit.name} with your structured answer."})
                continue
            run.final_text, run.stopped = reply.text, "end_turn"
            break
        for c in reply.tool_calls:
            if submit and c.name == submit.name:
                problem = validate(c.arguments) if validate else None
                if problem:
                    run.invalid_outputs.append(problem)
                    content = f"ERROR: invalid {submit.name}: {problem}. Fix it and call {submit.name} again."
                else:
                    run.submitted, content = c.arguments, "accepted"
                messages.append({"role": "tool", "tool_call_id": c.id, "content": content})
                continue
            say(role, f"-> {c.name}({', '.join(f'{k}={v}' for k, v in c.arguments.items())})")
            try:
                content = toolbox.call(c.name, c.arguments) if toolbox else "ERROR: no tools"
            except ToolNotAllowed as e:
                content = f"ERROR: ToolNotAllowed: {e}"
            run.calls.append({"tool": c.name, "args": c.arguments, "ok": not content.startswith("ERROR")})
            messages.append({"role": "tool", "tool_call_id": c.id, "content": content})
        if run.submitted is not None:
            run.final_text, run.stopped = reply.text, "submitted"
            break
        if len(run.invalid_outputs) >= 2:
            run.stopped = "invalid_output"
            break
    return run


In [ ]:
%%writefile netops/agents/single.py
"""Lab 1 -- ONE agent with every tool, including tools that change the network, and no gate."""
from __future__ import annotations

from netops.agents.loop import AgentRun, run_agent
from netops.agents.tools import READ_TOOLS, UNGATED_TOOLS, ToolBox
from netops.evidence import EvidenceStore
from netops.llm import LLM
from netops.network import Network

SYSTEM = """You are an autonomous network operations agent on call for an enterprise network.
Investigate the incident with your tools, fix the problem, and say clearly when the incident is resolved.
You have read tools and tools that change the network."""


class SingleAgent:
    name = "single-agent"

    def __init__(self, llm: LLM, network: Network, evidence: EvidenceStore | None = None,
                 extra_instructions: str = ""):
        # (`is not None`: an EMPTY EvidenceStore is falsy, and the caller's store must be the one we fill.)
        # extra_instructions: what most teams do after a bad incident -- add a line to the prompt.
        # Lab 1's stretch cell shows why that does not transfer to the NEXT incident.
        self.llm, self.net, self.evidence = llm, network, evidence if evidence is not None else EvidenceStore()
        self.extra_instructions = extra_instructions.strip()

    @property
    def system(self) -> str:
        return SYSTEM + (f"\n{self.extra_instructions}" if self.extra_instructions else "")

    def run(self, alert: str, trace=None) -> AgentRun:
        tools = ToolBox(self.net, self.evidence, list(READ_TOOLS) + list(UNGATED_TOOLS), agent=self.name, ungated=True)
        return run_agent(self.llm, role=self.name, system=self.system, task=alert, toolbox=tools, max_turns=12, trace=trace)


In [ ]:
%%writefile netops/agents/tools.py
"""The only way an agent touches the network. Allowlists are enforced HERE, in code."""
from __future__ import annotations

from typing import Callable, Iterable

from netops.evidence import EvidenceStore
from netops.llm import ToolSpec
from netops.network import Network


class ToolNotAllowed(PermissionError):
    pass


def _schema(props: dict, required: list[str]) -> dict:
    return {"type": "object", "properties": props, "required": required, "additionalProperties": False}


_D = {"type": "string", "description": "device hostname, e.g. BRANCH-7"}
_I = {"type": "string", "description": "interface, e.g. Ethernet1"}

READ_TOOLS: dict[str, tuple[str, dict, Callable]] = {
    "ospf_neighbors": ("OSPF neighbor table (show ip ospf neighbor). Empty list = no adjacency.",
                       _schema({"device": _D}, ["device"]), lambda n, a: n.ospf_neighbors(a["device"])),
    "ospf_interface": ("OSPF settings of an interface: area, hello/dead timers (show ip ospf interface).",
                       _schema({"device": _D, "interface": _I}, ["device", "interface"]),
                       lambda n, a: n.ospf_interface(a["device"], a["interface"])),
    "interface": ("Interface status and link-status-change counter (show interfaces).",
                  _schema({"device": _D, "interface": _I}, ["device", "interface"]),
                  lambda n, a: n.interface(a["device"], a["interface"])),
    "route": ("Look up a prefix in the routing table (show ip route).",
              _schema({"device": _D, "prefix": {"type": "string"}}, ["device", "prefix"]),
              lambda n, a: n.route(a["device"], a["prefix"])),
    "ping": ("Ping from a device, optionally from a source address.",
             _schema({"device": _D, "destination": {"type": "string"}, "source": {"type": "string"}},
                     ["device", "destination"]),
             lambda n, a: n.ping(a["device"], a["destination"], a.get("source"))),
    "logs": ("The full syslog buffer (show logging). Can be hundreds of lines.",
             _schema({"device": _D}, ["device"]), lambda n, a: n.logs(a["device"])),
    "log_summary": ("Syslog summarised: counts per category plus the OSPF/CONFIG lines verbatim.",
                    _schema({"device": _D}, ["device"]), lambda n, a: n.log_summary(a["device"])),
}

# Lab 1 ONLY. These change the network with no playbook, no gate, no human. That is the point of Lab 1.
UNGATED_TOOLS: dict[str, tuple[str, dict, Callable]] = {
    "bounce_interface": ("shutdown / no shutdown an interface. Applies immediately.",
                         _schema({"device": _D, "interface": _I}, ["device", "interface"]),
                         lambda n, a: (n.configure(a["device"], [f"interface {a['interface']}", "shutdown"], by="single-agent"),
                                       n.configure(a["device"], [f"interface {a['interface']}", "no shutdown"],
                                                   by="single-agent"))[-1].applied),
    "clear_counters": ("clear counters on an interface. Applies immediately.",
                       _schema({"device": _D, "interface": _I}, ["device", "interface"]),
                       lambda n, a: n.clear_counters(a["device"], a["interface"]) or "counters cleared"),
}


class ToolBox:
    def __init__(self, network: Network, evidence: EvidenceStore, allow: Iterable[str], *, agent: str,
                 ungated: bool = False):
        self.net, self.evidence, self.agent = network, evidence, agent
        self.table = dict(READ_TOOLS) | (dict(UNGATED_TOOLS) if ungated else {})
        self.allow = list(dict.fromkeys(allow))
        unknown = [t for t in self.allow if t not in self.table]
        if unknown:
            raise ValueError(f"{agent}: unknown or ungated tools without ungated=True: {unknown}")
        self.calls: list[dict] = []

    def specs(self) -> list[ToolSpec]:
        return [ToolSpec(name=t, description=self.table[t][0], parameters=self.table[t][1]) for t in self.allow]

    def call(self, name: str, args: dict) -> str:
        if name not in self.allow:
            raise ToolNotAllowed(f"{self.agent} may not call {name}")
        try:
            result = self.table[name][2](self.net, args)
        except KeyError as e:
            return f"ERROR: missing argument {e}"
        except Exception as e:                          # device errors, bad interface names...
            return f"ERROR: {e}"
        ev = self.evidence.add(name, args, result)
        self.calls.append({"tool": name, "args": args, "ev": ev.id})
        return self.evidence.text(ev)


In [ ]:
%%writefile netops/agents/troubleshooting.py
"""Troubleshooting specialist: READ-ONLY. Runs the runbook, detects faults deterministically,
lets the model rank and narrate them in a validated structure. It can't drop or invent a finding."""
from __future__ import annotations

import json
from importlib import resources

import yaml
from pydantic import BaseModel, Field

from netops.agents.loop import run_agent
from netops.agents.tools import ToolBox
from netops.evidence import EvidenceStore
from netops.llm import LLM, ToolSpec
from netops.network import Network
from netops.playbook import load_playbook
from netops.protocol import Message

ALLOW = ["ping", "route", "ospf_neighbors", "ospf_interface", "interface", "log_summary"]
FLAP_THRESHOLD = 4

SYSTEM = """ROLE: tier-2 troubleshooting specialist in a NOC team.
YOU OWN: explaining why the branch lost service AND why it did not recover.
YOU MAY NOT: change anything -- your tools are read-only.
OUTPUT: call submit_diagnosis once. Keep every detected finding; rank the one that explains the outage first.
EVIDENCE RULE: cite evidence ids (ev_03) for every claim."""

SUBMIT = ToolSpec(name="submit_diagnosis", description="Submit the ranked findings with a short narrative each.",
                  parameters={"type": "object", "properties": {
                      "findings": {"type": "array", "items": {"type": "object", "properties": {
                          "id": {"type": "string"}, "evidence_ids": {"type": "array", "items": {"type": "string"}},
                          "narrative": {"type": "string"}}, "required": ["id", "narrative"]}},
                      "summary": {"type": "string"}}, "required": ["findings", "summary"]})


class Finding(BaseModel):
    id: str                       # a playbook entry id when one matches
    title: str
    explains_outage: bool
    evidence_ids: list[str]
    observed: dict = Field(default_factory=dict)
    narrative: str = ""


def topology_params(net: Network, device: str = "BRANCH-7") -> dict:
    up = net.uplink(device)
    control = next(d for d, v in net.spec["devices"].items() if v["role"] == "branch" and d != device)
    return {"device": device, "uplink": up["interface"], "peer": up["peer"], "peer_interface": up["peer_interface"],
            "service": net.service["address"], "service_prefix": net.service["prefix"],
            "lan_gateway": net.lan_gateway(device), "control": control}


def run_runbook(tools: ToolBox, params: dict) -> dict[str, str]:
    """Run diagnose.yaml in order. Returns {step_key: evidence_id}."""
    skill = yaml.safe_load(resources.files("netops").joinpath("data", "diagnose.yaml").read_text())
    got = {}
    for step in skill["steps"]:
        args = {k: v.format(**params) for k, v in step["args"].items()}
        tools.call(step["tool"], args)
        key = f"{step['tool']}:{args['device']}"
        got[key] = tools.calls[-1]["ev"]
    return got


def detect(ev: EvidenceStore, got: dict[str, str], p: dict) -> list[Finding]:
    """Deterministic detectors -- the outcome never depends on the model."""
    playbook = load_playbook()
    findings: list[Finding] = []
    intf = ev.get(got[f"interface:{p['device']}"]).result
    logs = ev.get(got[f"log_summary:{p['device']}"]).result
    if intf["link_status_changes"] > FLAP_THRESHOLD or logs["counts"].get("LINK", 0) > 20:
        e = playbook["interface_flap"]
        findings.append(Finding(id=e.id, title=e.title, explains_outage=e.root_cause,
                                evidence_ids=[got[f"interface:{p['device']}"], got[f"log_summary:{p['device']}"]],
                                observed={"link_status_changes": intf["link_status_changes"],
                                          "link_log_lines": logs["counts"].get("LINK", 0)}))
    nbrs = ev.get(got[f"ospf_neighbors:{p['device']}"]).result
    full = any(n["interface"] == p["uplink"] and n["state"] == "full" for n in nbrs)
    local = ev.get(got[f"ospf_interface:{p['device']}"]).result
    peer = ev.get(got[f"ospf_interface:{p['peer']}"]).result
    if not full:
        base = [got[f"ospf_neighbors:{p['device']}"], got[f"ospf_interface:{p['device']}"],
                got[f"ospf_interface:{p['peer']}"]]
        if local["area"] != peer["area"]:
            e = playbook["ospf_area_mismatch"]
            findings.append(Finding(id=e.id, title=e.title, explains_outage=e.root_cause, evidence_ids=base,
                                    observed={"uplink": p["uplink"], "local_area": local["area"], "peer": p["peer"],
                                              "peer_interface": p["peer_interface"], "peer_area": peer["area"]}))
        else:
            findings.append(Finding(id="ospf_adjacency_down_unknown", title="OSPF adjacency not FULL, cause unknown",
                                    explains_outage=True, evidence_ids=base,
                                    observed={"states": [n["state"] for n in nbrs]}))
    return findings


class TroubleshootingAgent:
    name = "troubleshooting"

    def __init__(self, llm: LLM, network: Network, evidence: EvidenceStore, trace=None):
        self.llm, self.net, self.ev, self.trace = llm, network, evidence, trace or (lambda *a: None)
        self.last_run = None

    def diagnose(self, task: str) -> tuple[list[Finding], str]:
        p = topology_params(self.net)
        tools = ToolBox(self.net, self.ev, ALLOW, agent=self.name)
        self.trace(self.name, "running the runbook: ping -> route -> neighbors -> BOTH ends -> interface -> logs -> control")
        got = run_runbook(tools, p)
        found = detect(self.ev, got, p)
        detected = [{"id": f.id, "title": f.title, "explains_outage": f.explains_outage, "evidence_ids": f.evidence_ids,
                     "observed": f.observed} for f in found]
        evidence_text = "\n".join(self.ev.text(i, limit=600) for i in got.values())
        prompt = (f"{task}\n\nRunbook evidence:\n{evidence_text}\n\nDetectors found: {json.dumps(detected)}\n"
                  f"Rank them (the one that explains the outage first) and write one narrative line each.\n"
                  f"CONTEXT_JSON: {json.dumps({'detected': detected})}")
        ids = {f.id for f in found}

        def validate(a: dict) -> str | None:
            got_ids = [x.get("id") for x in a.get("findings", [])]
            if set(got_ids) != ids:
                return f"findings must be exactly {sorted(ids)}, got {got_ids}"
            bad = [e for x in a["findings"] for e in x.get("evidence_ids", []) if e not in self.ev]
            return f"unknown evidence ids {bad}" if bad else None

        run = run_agent(self.llm, role=self.name, system=SYSTEM, task=prompt, submit=SUBMIT, validate=validate,
                        trace=self.trace)
        self.last_run = run
        if run.submitted:
            order = [x["id"] for x in run.submitted["findings"]]
            notes = {x["id"]: x["narrative"] for x in run.submitted["findings"]}
            found = sorted(found, key=lambda f: order.index(f.id))
            for f in found:
                f.narrative = notes[f.id]
            return found, run.submitted["summary"]
        self.trace(self.name, "model output invalid -> deterministic findings kept as detected")
        for f in found:
            f.narrative = f"{f.title} (deterministic detector)"
        return sorted(found, key=lambda f: not f.explains_outage), "deterministic diagnosis"

    def neighbor_evidence(self, device: str, peer: str) -> list[str]:
        """Blast-radius evidence: the neighbor tables on BOTH ends of the link."""
        tools = ToolBox(self.net, self.ev, ["ospf_neighbors"], agent=self.name)
        for d in (device, peer):
            tools.call("ospf_neighbors", {"device": d})
        return [c["ev"] for c in tools.calls]

    def handle(self, msg: Message) -> Message:
        if msg.payload.get("want") == "neighbor_evidence":
            ids = self.neighbor_evidence(msg.payload["device"], msg.payload["peer"])
            return Message(**{"from": self.name}, to=msg.from_, intent="diagnose", task=msg.task,
                           payload={"evidence_ids": ids}, status="done")
        findings, summary = self.diagnose(msg.task)
        return Message(**{"from": self.name}, to=msg.from_, intent="diagnose", task=msg.task,
                       payload={"findings": [f.model_dump() for f in findings], "summary": summary}, status="done")


In [ ]:
%%writefile netops/agents/validation.py
"""Validation specialist: independent, read-only, no ego in the fix.
  pre  -- before a human sees it: is the proposal exactly what the playbook allows? is the fault still live?
          is the blast radius backed by neighbor tables on BOTH ends?
  post -- after apply: poll until expected_post_state holds (FULL + reachable) or the deadline passes.
          (Real cEOS: FULL arrives 3-9 s after the fix, the route 9-16 s. One immediate read would lie.)"""
from __future__ import annotations

from netops.agents.tools import ToolBox
from netops.evidence import EvidenceStore
from netops.network import Network
from netops.playbook import CommandGate, PlaybookError, load_playbook, render
from netops.protocol import Message

ALLOW = ["ospf_neighbors", "ospf_interface", "route", "ping"]


class ValidationAgent:
    name = "validation"

    def __init__(self, network: Network, evidence: EvidenceStore, trace=None, deadline_s: float = 60, poll_s: float = 3):
        self.net, self.ev, self.trace = network, evidence, trace or (lambda *a: None)
        self.deadline_s, self.poll_s = deadline_s, poll_s
        self.playbook = load_playbook()

    def _tools(self) -> ToolBox:
        return ToolBox(self.net, self.ev, ALLOW, agent=self.name)

    def pre_check(self, prop: dict) -> list[dict]:
        checks = []
        entry = self.playbook.get(prop["playbook_entry"])

        def add(name, ok, detail):
            checks.append({"check": name, "ok": bool(ok), "detail": detail})

        add("playbook_entry_exists", entry is not None, prop["playbook_entry"])
        if entry is None:
            return checks
        _, rejected = CommandGate(entry.allowed_commands).split(prop["commands"])
        add("commands_in_allowed_commands", not rejected, "all allowed" if not rejected else str([r.command for r in rejected]))
        try:
            same = render(entry, prop["params"]) == prop["commands"]
        except PlaybookError as e:
            same = False
        add("commands_equal_playbook_render", same, "commands are exactly the rendered playbook fix")
        add("expected_post_state_matches", prop["expected_post_state"] == entry.expected_post_state,
            "; ".join(entry.expected_post_state))
        t = self._tools()
        dev, up = prop["device"], prop["params"]["uplink"]
        peer = self.net.uplink(dev)
        t.call("ospf_interface", {"device": dev, "interface": up})
        t.call("ospf_interface", {"device": peer["peer"], "interface": peer["peer_interface"]})
        local, remote = (self.ev.get(c["ev"]).result for c in t.calls[-2:])
        add("fault_still_present", local["area"] != remote["area"],
            f"{dev} {up} area {local['area']} vs {peer['peer']} area {remote['area']}")
        add("target_matches_neighbor", prop["params"]["expected_area"] == remote["area"],
            f"expected_area {prop['params']['expected_area']} vs neighbor {remote['area']}")
        cited = [i for i in prop.get("blast_radius_evidence_ids", []) if i in self.ev]
        seen = {self.ev.get(i).args.get("device") for i in cited if self.ev.get(i).tool == "ospf_neighbors"}
        need = {dev, peer["peer"]}
        add("blast_radius_evidenced", need <= seen,
            "neighbor tables on both ends cited" if need <= seen else
            f"no neighbor-table evidence for {sorted(need - seen)} -- what else rides on this link?")
        return checks

    def post_check(self, prop: dict) -> list[dict]:
        dev, up = prop["device"], prop["params"]["uplink"]
        waited = 0.0
        while True:
            t = self._tools()
            t.call("ospf_neighbors", {"device": dev})
            nbrs = self.ev.get(t.calls[-1]["ev"]).result
            full = any(n["interface"] == up and n["state"] == "full" for n in nbrs)
            t.call("ping", {"device": dev, "destination": self.net.service["address"],
                            "source": self.net.lan_gateway(dev)})
            ping = self.ev.get(t.calls[-1]["ev"]).result
            reachable = ping["sent"] > 0 and ping["received"] == ping["sent"]
            if (full and reachable) or waited >= self.deadline_s:
                break
            self.net.wait(self.poll_s)
            waited += self.poll_s
        self.trace(self.name, f"after {waited:.0f}s: OSPF {'FULL' if full else 'not FULL'}, "
                              f"reachable={reachable} ({ping['received']}/{ping['sent']})")
        return [{"check": "ospf neighbor state = FULL", "ok": full, "detail": f"{dev} {up}"},
                {"check": "branch l3 reachable = true", "ok": reachable,
                 "detail": f"{ping['received']}/{ping['sent']} after {waited:.0f}s"}]

    def handle(self, msg: Message) -> Message:
        prop = msg.payload["proposal"]
        checks = self.pre_check(prop) if msg.payload.get("phase") == "pre" else self.post_check(prop)
        return Message(**{"from": self.name}, to=msg.from_, intent="validate", task=msg.task,
                       payload={"phase": msg.payload.get("phase", "post"), "checks": checks,
                                "passed": all(c["ok"] for c in checks)}, status="done")


In [ ]:
%%writefile netops/approval.py
"""Human approval as code: a single-use token bound to the EXACT proposal, minted only on a human's action.

The Executor refuses to apply anything without a valid token. If anyone edits the proposal after approval,
its digest changes and the token no longer fits. A used token can never be used again.
"""
from __future__ import annotations

import hashlib
import hmac
import json
import secrets
import time
import uuid
from dataclasses import dataclass, field
from typing import Callable


class GateError(PermissionError):
    pass


def digest_of(proposal: dict) -> str:
    return hashlib.sha256(json.dumps(proposal, sort_keys=True, default=str).encode()).hexdigest()


@dataclass(frozen=True)
class ApprovalToken:
    digest: str
    approver: str
    run_id: str
    expiry: float
    mac: str = field(repr=False)

    def __repr__(self) -> str:
        return f"ApprovalToken(approver={self.approver!r}, digest={self.digest[:8]}..., mac=[redacted])"

    __str__ = __repr__


class Approvals:
    def __init__(self, ttl_s: int = 900, clock: Callable[[], float] = time.time, secret: bytes | None = None,
                 run_id: str | None = None):
        self._secret = secret or secrets.token_bytes(32)
        self.run_id = run_id or uuid.uuid4().hex[:8]
        self._ttl, self._clock = ttl_s, clock
        self._used: set[str] = set()
        self.log: list[dict] = []

    def _mac(self, digest: str, approver: str, run_id: str, expiry: float) -> str:
        return hmac.new(self._secret, f"{digest}|{approver}|{run_id}|{expiry!r}".encode(), hashlib.sha256).hexdigest()

    def approve(self, proposal: dict, approver: str) -> ApprovalToken:
        d, exp = digest_of(proposal), self._clock() + self._ttl
        self.log.append({"event": "approved", "approver": approver, "digest": d[:12]})
        return ApprovalToken(d, approver, self.run_id, exp, self._mac(d, approver, self.run_id, exp))

    def consume(self, token: ApprovalToken | None, proposal: dict) -> None:
        if not isinstance(token, ApprovalToken):
            raise GateError("no approval: a human must approve this change first")
        if not hmac.compare_digest(self._mac(token.digest, token.approver, token.run_id, token.expiry), token.mac):
            raise GateError("invalid approval signature")
        if token.run_id != self.run_id:
            raise GateError("approval belongs to a different run")
        if token.digest != digest_of(proposal):
            raise GateError("the proposal changed after it was approved")
        if self._clock() > token.expiry:
            raise GateError("approval expired")
        if token.mac in self._used:
            raise GateError("approval already used")
        self._used.add(token.mac)
        self.log.append({"event": "consumed", "approver": token.approver})


In [ ]:
%%writefile netops/checklist.py
"""The on-call engineer's nine questions before ANY change to a production router.

These are the questions a senior network engineer asks out loud before typing `configure`. The gate panel
(`render.gate`) prints them next to the proposal. Some the kit can answer from evidence (a pre-check, a field of the
proposal, a property of the executor). The rest stay with the human -- and the panel says so, instead of pretending.

    status "kit"   -> answered by code: a validation pre-check, the proposal itself, or the executor
    status "human" -> only the approver can answer it (change window, business context, judgement)
"""
from __future__ import annotations

# (key, question). Order matters: it is the order an engineer walks through a change.
QUESTIONS: list[tuple[str, str]] = [
    ("evidence", "Does the evidence support the diagnosis? (area 0.0.0.1 on BRANCH-7 vs 0.0.0.0 on HUB-1, in actual output)"),
    ("target", "Is the target value verified against the neighbor -- confirmed, not remembered?"),
    ("scope", "Scoped to only the affected interface?"),
    ("blast_radius", "Blast radius: what else rides on this link?"),
    ("rollback", "Tested rollback in hand before we commit?"),
    ("window", "Are we allowed to touch routing right now (window / approval context)?"),
    ("pass_fail", "Pass/fail defined before commit: FULL, route back, 5/5 ping?"),
    ("backout", "Back-out trigger set: not FULL in ~60 s -> revert?"),
    ("save", "Save only after validation passes -- write memory is the last step."),
]

# Plan 3 contract name: the nine questions as plain strings (guide, approver sheet, cheat sheet, deck).
NOC_CHECKLIST: list[str] = [q for _, q in QUESTIONS]

SAVE_COMMANDS = ("write memory", "copy running-config startup-config", "wr mem")


def answer(gate) -> list[dict]:
    """[{n, question, status: kit|human, ok: bool|None, detail}] for a GateRequest (or anything with
    .proposal and .pre_checks). `ok is None` means: the human must answer this one."""
    p = gate.proposal
    pre = {c["check"]: c for c in gate.pre_checks}

    def from_check(name: str) -> tuple[str, bool | None, str]:
        c = pre.get(name)
        return ("kit", c["ok"], f"pre-check {name}: {c['detail']}") if c else ("human", None, "no pre-check ran")

    interfaces = [c for c in p["commands"] if c.startswith("interface ")]
    saves = [c for c in p["commands"] if c.startswith(SAVE_COMMANDS)]
    rows = {
        "evidence": from_check("fault_still_present"),
        "target": from_check("target_matches_neighbor"),
        "scope": ("kit", len(interfaces) == 1, f"interfaces touched: {', '.join(interfaces) or 'none'}"),
        "blast_radius": from_check("blast_radius_evidenced"),
        "rollback": ("kit", bool(p.get("rollback")), " / ".join(p.get("rollback", [])) or "no rollback"),
        "window": ("human", None, "the kit cannot know your change calendar -- you answer this"),
        "pass_fail": from_check("expected_post_state_matches"),
        "backout": ("kit", True, "executor backs out automatically if OSPF is not FULL before its timer"),
        "save": ("kit", not saves, "no save command in the change; save after post-validation"
                 if not saves else f"save inside the change: {saves}"),
    }
    out = []
    for n, (key, question) in enumerate(QUESTIONS, 1):
        status, ok, detail = rows[key]
        out.append({"n": n, "key": key, "question": question, "status": status, "ok": ok, "detail": detail})
    return out


In [ ]:
%%writefile netops/data/diagnose.yaml
# the on-call engineer's ordered checks for "a branch lost the core service". The ORDER is the engineer's runbook:
# outcome first (ping, route), then the adjacency, then BOTH ends of the link, then the noise, then the
# healthy control site to diff against. The model reasons over this evidence; it does not pick the commands.
skill: diagnose_branch_unreachable
steps:
  - {tool: ping,           args: {device: "{device}", destination: "{service}", source: "{lan_gateway}"}}
  - {tool: route,          args: {device: "{device}", prefix: "{service_prefix}"}}
  - {tool: ospf_neighbors, args: {device: "{device}"}}
  - {tool: ospf_interface, args: {device: "{device}", interface: "{uplink}"}}
  - {tool: ospf_interface, args: {device: "{peer}", interface: "{peer_interface}"}}
  - {tool: interface,      args: {device: "{device}", interface: "{uplink}"}}
  - {tool: log_summary,    args: {device: "{device}"}}
  - {tool: ospf_neighbors, args: {device: "{control}"}}


In [ ]:
%%writefile netops/data/incident.yaml
# The ONE incident the workshop chases. Everything the simulator does comes from this file.
id: branch7_ospf_area_mismatch
title: "Branch-7 lost the core service"
start_time: "2026-10-12T03:14:00"          # the cold open: 3:14 am
seed: 7                                     # all randomness (timing jitter) is seeded
service: {prefix: 10.100.10.0/24, address: 10.100.10.10}
devices:
  BRANCH-7: {role: branch, router_id: 1.1.1.11, lan: 10.20.20.0/24, lan_gateway: 10.20.20.1}
  HUB-1:    {role: hub,    router_id: 1.1.1.1,  originates_service: true}
  BRANCH-8: {role: branch, router_id: 1.1.1.12, lan: 10.30.30.0/24, lan_gateway: 10.30.30.1}   # healthy control
  HUB-2:    {role: hub,    router_id: 2.2.2.2,  originates_service: true}
links:
  - {a: "BRANCH-7:Ethernet1", b: "HUB-1:Ethernet1", subnet: 10.0.12.0/31, area: 0.0.0.0,
     a_description: "uplink to HUB-1", b_description: "downlink to BRANCH-7"}
  - {a: "BRANCH-8:Ethernet1", b: "HUB-2:Ethernet1", subnet: 10.0.22.0/31, area: 0.0.0.0,
     a_description: "uplink to HUB-2", b_description: "downlink to BRANCH-8"}
timers: {hello: 10, dead: 40}
# Measured on ceos:4.34.8M with 10/40 timers (2026-10-12): FULL 3-9 s after a fix, route 9-16 s.
convergence: {full_after_s: 9.0, route_after_full_s: 6.5}
fault:                                      # minutes relative to start_time, applied in order
  - {at_min: -60, op: flap,     link: "BRANCH-7:Ethernet1", count: 6,  every_s: 20}   # flapped and RECOVERED
  - {at_min: -5,  op: set_area, link: "BRANCH-7:Ethernet1", area: 0.0.0.1}           # the silent root cause
  - {at_min: -4,  op: flap,     link: "BRANCH-7:Ethernet1", count: 12, every_s: 15}   # flapped and did NOT recover
log_volume: 400                             # BRANCH-7 log lines; padded with further (genuine) flaps
variants:
  mtu_mismatch:                             # a different silent fault (neighbor stuck in exchStart)
    fault:
      - {at_min: -60, op: flap,    link: "BRANCH-7:Ethernet1", count: 6,  every_s: 20}
      - {at_min: -5,  op: set_mtu, link: "BRANCH-7:Ethernet1", mtu: 9000}
      - {at_min: -4,  op: flap,    link: "BRANCH-7:Ethernet1", count: 12, every_s: 15}
  full_no_route:                            # adjacency FULL but the route is filtered (validator trap)
    fault:
      - {at_min: -5,  op: filter_route, device: BRANCH-7}


In [ ]:
%%writefile netops/device/__init__.py
"""Device layer: one DeviceDriver interface, two backends (simulated cEOS, real cEOS over eAPI)."""


In [ ]:
%%writefile netops/device/ceos.py
"""CeosDevice -- a real Arista cEOS (or EOS) over eAPI (JSON-RPC over HTTP). Standard library only.

Same DeviceDriver interface as SimulatedDevice, same commands, same JSON -> the SAME agent code runs unchanged.
Configuration goes through a configure session + commit, so a change applies atomically.
"""
from __future__ import annotations

import base64
import json
import time
import urllib.request
import uuid
from typing import Callable, Literal

from netops.device.driver import CommandError, ConfigResult

Transport = Callable[[dict], dict]       # takes a JSON-RPC request, returns the JSON-RPC response


class CeosDevice:
    def __init__(self, hostname: str, host: str, username: str = "admin", password: str = "admin",
                 port: int = 80, https: bool = False, timeout: float = 15, transport: Transport | None = None):
        self.hostname = hostname
        if ":" in host:                                   # "clab.orb.local:8011" style
            host, port = host.rsplit(":", 1)[0], int(host.rsplit(":", 1)[1])
        self.url = f"{'https' if https else 'http'}://{host}:{port}/command-api"
        self._auth = base64.b64encode(f"{username}:{password}".encode()).decode()
        self.timeout = timeout
        self._transport = transport or self._http

    def _http(self, request: dict) -> dict:
        req = urllib.request.Request(self.url, data=json.dumps(request).encode(),
                                     headers={"Content-Type": "application/json", "Authorization": f"Basic {self._auth}"})
        with urllib.request.urlopen(req, timeout=self.timeout) as resp:
            return json.loads(resp.read())

    def _rpc(self, cmds: list[str], fmt: str) -> list[dict]:
        # eAPI sessions start unprivileged; ping-with-source and config sessions need `enable` (verified on 4.34.8M).
        request = {"jsonrpc": "2.0", "method": "runCmds", "id": uuid.uuid4().hex[:8],
                   "params": {"version": 1, "cmds": ["enable", *cmds], "format": fmt}}
        response = self._transport(request)
        if "error" in response:
            err = response["error"]
            details = [d.get("errors") for d in err.get("data", []) if isinstance(d, dict) and d.get("errors")]
            raise CommandError(f"{self.hostname}: {err.get('message')} {details or ''}".strip())
        return response["result"][1:]                     # drop the result of `enable`

    def run(self, cmds: list[str], fmt: Literal["json", "text"] = "json") -> list[dict]:
        return self._rpc(cmds, fmt)

    def configure(self, cmds: list[str], *, by: str = "operator") -> ConfigResult:
        session = f"netops-{uuid.uuid4().hex[:6]}"
        self._rpc([f"configure session {session}", *cmds, "commit"], "json")
        return ConfigResult(device=self.hostname, applied=list(cmds), by=by)

    def wait(self, seconds: float) -> None:
        time.sleep(seconds)


In [ ]:
%%writefile netops/device/driver.py
"""The DeviceDriver interface — the seam that lets the SAME agent code run on a simulated switch
(for students) and on a real Arista cEOS over eAPI (instructor demo).

It mirrors eAPI on purpose:
  run(["show ip ospf neighbor"], fmt="json")  -> [ {...the exact JSON cEOS returns...} ]
  run(["show logging"], fmt="text")           -> [ {"output": "...the exact text..."} ]
  configure(["interface Ethernet1", "ip ospf area 0.0.0.0"], by="on-call")
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Literal, Protocol


class CommandError(RuntimeError):
    """The device rejected a command (EOS prints '% Invalid input')."""


@dataclass
class ConfigResult:
    device: str
    applied: list[str] = field(default_factory=list)
    by: str = "operator"


class DeviceDriver(Protocol):
    hostname: str

    def run(self, cmds: list[str], fmt: Literal["json", "text"] = "json") -> list[dict]: ...

    def configure(self, cmds: list[str], *, by: str = "operator") -> ConfigResult: ...

    def wait(self, seconds: float) -> None:
        """Let time pass (virtual on the simulator, real on hardware). Used for convergence polling."""
        ...


In [ ]:
%%writefile netops/device/fabric.py
"""The incident as a state machine (the "physics" behind the simulated switches).

Truth table, kept deliberately small and technically correct:
  * An OSPF adjacency can form only if both ends are up, not passive, in the SAME area, with the
    SAME hello/dead timers. Area mismatch => hellos are dropped => the neighbor is ABSENT
    (verified on cEOS 4.34.8M). MTU mismatch => the neighbor is stuck in exchStart (that's MTU, not area).
  * A compatible adjacency becomes FULL `full_after_s` after the link/config settles, and routes
    learned over it install `route_after_full_s` later (SPF hold) -- the real "FULL but no route yet" window.
  * A device reaches the service prefix iff it originates it, or it has a FULL adjacency whose routes
    have installed to a hub that originates it (and the route isn't filtered).

Logs are produced BY state transitions (never hand-scripted): flaps before the area change recover
(adjacency re-established), flaps after it never do. The absence of recovery is the real tell.
"""
from __future__ import annotations

import ipaddress
import random
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from importlib import resources
from typing import Any

import yaml


def load_incident(variant: str | None = None) -> dict:
    """Load the incident definition (optionally a variant that replaces the fault list)."""
    with resources.files("netops").joinpath("data", "incident.yaml").open() as f:
        spec = yaml.safe_load(f)
    if variant:
        spec["fault"] = spec["variants"][variant]["fault"]
        spec["variant"] = variant
    return spec


@dataclass
class Port:
    """One end of a point-to-point link."""
    device: str
    name: str
    ip: str
    peer_device: str
    peer_port: str
    peer_ip: str
    description: str
    area: str
    hello: int
    dead: int
    mtu: int = 1500
    passive: bool = False
    admin_up: bool = True
    oper_up: bool = True
    link_status_changes: int = 0
    in_errors: int = 0
    last_change: datetime | None = None


@dataclass
class LinkTiming:
    """When the adjacency on a link became/will become FULL, and when its routes install."""
    full_at: datetime | None
    route_at: datetime | None


@dataclass
class LogEntry:
    ts: datetime
    device: str
    process: str      # "Ebra", "Ospf", "ConfigAgent" -- as cEOS prints them
    text: str
    category: str     # LINK / OSPF / CONFIG


class Fabric:
    """All simulated devices share one fabric: links, timing, logs and a virtual clock."""

    def __init__(self, variant: str | None = None, inject: bool = True):
        self.spec: dict[str, Any] = load_incident(variant)
        self.rng = random.Random(self.spec["seed"])
        self.start = datetime.fromisoformat(self.spec["start_time"])
        self.clock = self.start
        self.conv = self.spec["convergence"]
        self.ports: dict[tuple[str, str], Port] = {}
        self.timing: dict[frozenset, LinkTiming] = {}
        self.logs: dict[str, list[LogEntry]] = {d: [] for d in self.spec["devices"]}
        self.route_filters: set[str] = set()       # devices whose inbound service route is filtered
        self.saved: dict[str, bool] = {d: True for d in self.spec["devices"]}
        self.audit: list[dict] = []                # every configuration change, who made it, when
        self._build_links()
        if inject:
            self._inject_faults()

    # ------------------------------------------------------------------ topology
    def _build_links(self) -> None:
        t = self.spec["timers"]
        long_ago = self.start - timedelta(days=30)
        for link in self.spec["links"]:
            (ad, ap), (bd, bp) = link["a"].split(":"), link["b"].split(":")
            net = ipaddress.ip_network(link["subnet"])
            aip, bip = str(net.network_address), str(net.network_address + 1)
            self.ports[(ad, ap)] = Port(ad, ap, aip, bd, bp, bip, link["a_description"], link["area"],
                                        t["hello"], t["dead"], last_change=long_ago)
            self.ports[(bd, bp)] = Port(bd, bp, bip, ad, ap, aip, link["b_description"], link["area"],
                                        t["hello"], t["dead"], last_change=long_ago)
            self.timing[frozenset({(ad, ap), (bd, bp)})] = LinkTiming(full_at=long_ago, route_at=long_ago)

    def port(self, device: str, name: str) -> Port:
        try:
            return self.ports[(device, name)]
        except KeyError:
            raise KeyError(f"{device} has no interface {name}") from None

    def peer(self, p: Port) -> Port:
        return self.ports[(p.peer_device, p.peer_port)]

    def ports_of(self, device: str) -> list[Port]:
        return [p for (d, _), p in self.ports.items() if d == device]

    def _key(self, p: Port) -> frozenset:
        return frozenset({(p.device, p.name), (p.peer_device, p.peer_port)})

    # ------------------------------------------------------------------ protocol truth
    def compatible(self, p: Port) -> bool:
        """Can these two ends form an adjacency at all (ignoring time)? MTU is checked separately."""
        q = self.peer(p)
        return (p.oper_up and q.oper_up and not p.passive and not q.passive
                and p.area == q.area and p.hello == q.hello and p.dead == q.dead)

    def adjacency_state(self, p: Port) -> str | None:
        """None = no neighbor entry at all (area/timer mismatch, link down)."""
        if not self.compatible(p):
            return None
        if p.mtu != self.peer(p).mtu:
            return "exchStart"
        t = self.timing[self._key(p)]
        if t.full_at is not None and self.clock >= t.full_at:
            return "full"
        return "init"

    def routes_installed(self, p: Port) -> bool:
        t = self.timing[self._key(p)]
        return self.adjacency_state(p) == "full" and t.route_at is not None and self.clock >= t.route_at

    def originates(self, device: str, prefix: str) -> bool:
        dev = self.spec["devices"][device]
        if prefix == self.spec["service"]["prefix"]:
            return bool(dev.get("originates_service"))
        return dev.get("lan") == prefix

    def has_route(self, device: str, prefix: str) -> tuple[bool, Port | None]:
        """(present, via-port). Connected if originated locally; else learned over a FULL adjacency."""
        if self.originates(device, prefix):
            return True, None
        if prefix == self.spec["service"]["prefix"] and device in self.route_filters:
            return False, None
        for p in self.ports_of(device):
            if self.routes_installed(p) and self.originates(p.peer_device, prefix):
                return True, p
        return False, None

    def prefix_for(self, address: str) -> str | None:
        ip = ipaddress.ip_address(address)
        candidates = [self.spec["service"]["prefix"]] + [d["lan"] for d in self.spec["devices"].values() if "lan" in d]
        for pfx in candidates:
            if ip in ipaddress.ip_network(pfx):
                return pfx
        return None

    def can_ping(self, device: str, destination: str, source: str | None) -> bool:
        """Forward path (device -> destination prefix) and return path (destination owner -> source prefix)."""
        dst_pfx = self.prefix_for(destination)
        if dst_pfx is None or not self.has_route(device, dst_pfx)[0]:
            return False
        if source is None:
            return True
        src_pfx = self.prefix_for(source)
        owners = [d for d in self.spec["devices"] if self.originates(d, dst_pfx)]
        return src_pfx is not None and any(self.has_route(o, src_pfx)[0] for o in owners)

    # ------------------------------------------------------------------ time
    def advance(self, seconds: float) -> None:
        """Move the virtual clock, logging every adjacency that reaches FULL along the way."""
        end = self.clock + timedelta(seconds=seconds)
        pending = sorted((t.full_at, key) for key, t in self.timing.items()
                         if t.full_at is not None and self.clock < t.full_at <= end)
        for when, key in pending:
            self.clock = when
            a, b = (self.ports[k] for k in key)
            if self.adjacency_state(a) == "full":
                for p in (a, b):
                    q = self.peer(p)
                    self._log(p.device, "Ospf", "OSPF",
                              f"Instance 1: %OSPF-4-OSPF_ADJACENCY_ESTABLISHED: NGB {self._rid(q.device)}, "
                              f"interface {p.ip} adjacency established")
        self.clock = end

    def _rid(self, device: str) -> str:
        return self.spec["devices"][device]["router_id"]

    def _log(self, device: str, process: str, category: str, text: str) -> None:
        self.logs[device].append(LogEntry(self.clock, device, process, text, category))

    def _restart_adjacency(self, p: Port, was_full: bool) -> None:
        """Re-evaluate timing after any change on a link."""
        t = self.timing[self._key(p)]
        if self.compatible(p) and p.mtu == self.peer(p).mtu:
            if not was_full:
                t.full_at = self.clock + timedelta(seconds=self.conv["full_after_s"])
                t.route_at = t.full_at + timedelta(seconds=self.conv["route_after_full_s"])
        else:
            t.full_at = t.route_at = None

    # ------------------------------------------------------------------ events (used by faults AND by CLI)
    def set_oper(self, p: Port, up: bool) -> None:
        """A physical link state change hits BOTH ends of a point-to-point link."""
        was_full = self.adjacency_state(p) == "full"
        for x in (p, self.peer(p)):
            if x.oper_up == up:
                continue
            x.oper_up = up
            x.link_status_changes += 1
            x.in_errors += self.rng.randint(0, 2)
            x.last_change = self.clock
            self._log(x.device, "Ebra", "LINK", f"%LINEPROTO-5-UPDOWN: Line protocol on Interface {x.name} "
                                                f"({x.description}), changed state to {'up' if up else 'down'}")
            if not up and was_full:
                self._log(x.device, "Ospf", "OSPF",
                          f"Instance 1: %OSPF-4-OSPF_ADJACENCY_TEARDOWN: NGB {self._rid(self.peer(x).device)}, "
                          f"interface {x.ip} adjacency dropped: interface went down, state was: FULL")
        self._restart_adjacency(p, was_full and up)

    def flap(self, p: Port, count: int = 1, every_s: float = 15.0, down_s: float = 1.0) -> None:
        for _ in range(count):
            self.set_oper(p, False)
            self.advance(down_s)
            self.set_oper(p, True)
            self.advance(max(every_s - down_s, 0) + self.rng.uniform(0, 0.4))

    def configure_port(self, p: Port, attr: str, value: Any, by: str) -> None:
        """A configuration change (area, mtu, admin state...). Logged like cEOS's ConfigAgent."""
        was_full = self.adjacency_state(p) == "full"
        setattr(p, attr, value)
        self.saved[p.device] = False
        self._log(p.device, "ConfigAgent", "CONFIG", f"%SYS-5-CONFIG_I: Configured from console by {by} on vty3")
        self.audit.append({"ts": self.clock.isoformat(), "device": p.device, "interface": p.name,
                           "set": attr, "to": value, "by": by})
        if was_full and self.adjacency_state(p) != "full":
            for x in (p, self.peer(p)):
                self._log(x.device, "Ospf", "OSPF",
                          f"Instance 1: %OSPF-4-OSPF_STATE_CHANGE: NGB {self._rid(self.peer(x).device)} "
                          f"({self.peer(x).ip}) on interface {x.ip} changed from <FULL> to <DOWN>")
        self._restart_adjacency(p, was_full and self.adjacency_state(p) == "full")

    def set_admin(self, p: Port, up: bool, by: str) -> None:
        """`shutdown` / `no shutdown` on one end: the link goes down/up for both ends."""
        p.admin_up = up
        self.saved[p.device] = False
        self._log(p.device, "ConfigAgent", "CONFIG", f"%SYS-5-CONFIG_I: Configured from console by {by} on vty3")
        self.audit.append({"ts": self.clock.isoformat(), "device": p.device, "interface": p.name,
                           "set": "admin", "to": "up" if up else "down", "by": by})
        self.set_oper(p, up and self.peer(p).admin_up)

    def clear_counters(self, device: str, name: str | None, by: str) -> None:
        """`clear counters` — the flap counter goes to zero. The flap is 'gone'. Nothing is fixed."""
        for p in self.ports_of(device):
            if name is None or p.name == name:
                p.link_status_changes = 0
                p.in_errors = 0
        self.audit.append({"ts": self.clock.isoformat(), "device": device, "interface": name,
                           "set": "clear_counters", "to": 0, "by": by})

    # ------------------------------------------------------------------ the fault timeline
    def _inject_faults(self) -> None:
        events = sorted(self.spec["fault"], key=lambda e: e["at_min"])
        self.clock = self.start + timedelta(minutes=events[0]["at_min"])
        for e in events:
            target = self.start + timedelta(minutes=e["at_min"])
            if target > self.clock:
                self.advance((target - self.clock).total_seconds())
            if e["op"] == "filter_route":
                self.route_filters.add(e["device"])
                self._log(e["device"], "ConfigAgent", "CONFIG",
                          "%SYS-5-CONFIG_I: Configured from console by change-window-automation on vty3")
                continue
            p = self.port(*e["link"].split(":"))
            if e["op"] == "flap":
                self.flap(p, e["count"], e["every_s"])
            elif e["op"] == "set_area":
                self.configure_port(p, "area", e["area"], by="change-window-automation")
            elif e["op"] == "set_mtu":
                self.configure_port(p, "mtu", e["mtu"], by="change-window-automation")
            else:
                raise ValueError(f"unknown fault op {e['op']}")
        self._pad_logs(events)
        if self.clock < self.start:
            self.advance((self.start - self.clock).total_seconds())
        self.audit.clear()          # the fault is history, not something an agent or operator did today
        self.saved = {d: True for d in self.spec["devices"]}   # the bad config was saved in the change window

    def _pad_logs(self, events: list[dict]) -> None:
        """Keep flapping (genuinely) until BRANCH-7's buffer holds `log_volume` lines, without passing 03:14."""
        last = next((e for e in reversed(events) if e["op"] == "flap"), None)
        if not last:
            return
        p = self.port(*last["link"].split(":"))
        need = self.spec["log_volume"] - len(self.logs[p.device])
        if need <= 0:
            return
        flaps = -(-need // 2)                                   # 2 LINEPROTO lines per flap on a dead adjacency
        window = max((self.start - self.clock).total_seconds() - 2, flaps * 0.2)
        step = window / flaps
        for _ in range(flaps):
            self.set_oper(p, False)
            self.advance(step / 2)
            self.set_oper(p, True)
            self.advance(step / 2)


In [ ]:
%%writefile netops/device/simulator.py
"""SimulatedDevice — a fake Arista cEOS that answers like the real one.

Every JSON document below copies the key layout captured from ceos:4.34.8M on 2026-10-12
(see tests/fixtures/ceos/). The parsers in netops/parsers.py read real and simulated output with
the same code; tests/test_simulator.py checks the layouts match.
"""
from __future__ import annotations

import ipaddress
import re
from typing import Literal

from netops.device.driver import CommandError, ConfigResult
from netops.device.fabric import Fabric, Port

_LOG_HEADER = """Syslog logging: enabled
Buffer logging: level debugging
Console logging: level informational
Persistent logging: disabled
Monitor logging: level informational
Synchronous logging: disabled
Trap logging: level informational
Sequence numbers: disabled
Syslog facility: local4
Hostname format: Hostname only
Repeat logging interval: disabled
Repeat messages: disabled

Log Buffer:"""

_OPTIONS = {"multitopologyCapability": False, "externalRoutingCapability": True, "multicastCapability": False,
            "nssaCapability": False, "linkLocalSignaling": False, "demandCircuitsSupport": False,
            "opaqueLsaSupport": False, "doNotUseInRouteCalc": False}

_TEXT_ONLY = ("show logging", "ping", "show running-config")


def normalize_interface(name: str) -> str:
    """EOS accepts Et1 / eth1 / ethernet1 / Ethernet1 — so do we."""
    m = re.fullmatch(r"(?i)(et|eth|ethernet)\s*(\d+)", name.strip())
    if not m:
        raise CommandError(f"% Invalid input (interface '{name}')")
    return f"Ethernet{m.group(2)}"


class SimulatedDevice:
    def __init__(self, fabric: Fabric, hostname: str):
        if hostname not in fabric.spec["devices"]:
            raise ValueError(f"unknown device {hostname}")
        self.fabric, self.hostname = fabric, hostname

    # ------------------------------------------------------------------ DeviceDriver API
    def run(self, cmds: list[str], fmt: Literal["json", "text"] = "json") -> list[dict]:
        out = []
        for raw in cmds:
            cmd = " ".join(raw.split())
            if fmt == "json" and cmd.startswith(_TEXT_ONLY):
                raise CommandError(f"'{cmd}' has no JSON output on EOS; request it with fmt='text'")
            out.append(self._dispatch(cmd, fmt))
        return out

    def configure(self, cmds: list[str], *, by: str = "operator") -> ConfigResult:
        """Validate EVERY line first, then apply (atomic, like a cEOS configure session)."""
        plan, ctx = [], None
        for raw in cmds:
            line = " ".join(raw.split())
            if line in ("", "!", "configure", "configure terminal", "end", "exit", "commit") \
                    or line.startswith("configure session"):
                continue
            m = re.fullmatch(r"interface (\S+)", line)
            if m:
                ctx = self._port(normalize_interface(m.group(1)))
                continue
            if ctx is None:
                raise CommandError(f"% Invalid input (at '{line}')")
            if m := re.fullmatch(r"ip ospf area (\d+\.\d+\.\d+\.\d+)", line):
                ipaddress.IPv4Address(m.group(1))
                plan.append((ctx, "area", m.group(1)))
            elif line == "no ip ospf area":
                plan.append((ctx, "area", None))
            elif line in ("shutdown", "no shutdown"):
                plan.append((ctx, "admin", line == "no shutdown"))
            elif m := re.fullmatch(r"mtu (\d+)", line):
                plan.append((ctx, "mtu", int(m.group(1))))
            elif m := re.fullmatch(r"description (.+)", line):
                plan.append((ctx, "description", m.group(1)))
            elif line == "ip ospf network point-to-point":
                continue
            else:
                raise CommandError(f"% Invalid input (at '{line}')")
        for port, attr, value in plan:
            if attr == "admin":
                self.fabric.set_admin(port, value, by)
            else:
                self.fabric.configure_port(port, attr, value, by)
        return ConfigResult(device=self.hostname, applied=[l for l in cmds if l.strip()], by=by)

    def wait(self, seconds: float) -> None:
        self.fabric.advance(seconds)

    # ------------------------------------------------------------------ helpers
    def _port(self, name: str) -> Port:
        try:
            return self.fabric.port(self.hostname, name)
        except KeyError:
            raise CommandError(f"% Interface {name} does not exist") from None

    def _epoch(self, dt) -> float:
        return dt.timestamp() if dt else 0.0

    def _dispatch(self, cmd: str, fmt: str) -> dict:
        if cmd == "show ip ospf neighbor":
            return self._ospf_neighbor()
        if m := re.fullmatch(r"show ip ospf interface (\S+)", cmd):
            return self._ospf_interface(normalize_interface(m.group(1)))
        if m := re.fullmatch(r"show interfaces (\S+)", cmd):
            return self._interfaces(normalize_interface(m.group(1)))
        if m := re.fullmatch(r"show ip route (\S+)", cmd):
            return self._route(m.group(1))
        if m := re.fullmatch(r"clear counters(?: (\S+))?", cmd):
            self.fabric.clear_counters(self.hostname, normalize_interface(m.group(1)) if m.group(1) else None,
                                       by="cli")
            return {} if fmt == "json" else {"output": ""}
        if cmd == "show logging":
            return {"output": self._logging()}
        if m := re.fullmatch(r"ping (\S+)(?: source (\S+))?(?: repeat (\d+))?", cmd):
            return {"output": self._ping(m.group(1), m.group(2), int(m.group(3) or 5))}
        if m := re.fullmatch(r"show running-config interfaces (\S+)", cmd):
            return {"output": self._running(normalize_interface(m.group(1)))}
        raise CommandError(f"% Invalid input (at '{cmd}')")

    # ------------------------------------------------------------------ eAPI JSON (captured layouts)
    def _ospf_neighbor(self) -> dict:
        entries = []
        for p in self.fabric.ports_of(self.hostname):
            state = self.fabric.adjacency_state(p)
            if state is None:
                continue                                    # area mismatch: no entry at all
            q = self.fabric.peer(p)
            entries.append({
                "routerId": self.fabric.spec["devices"][q.device]["router_id"], "interfaceAddress": q.ip,
                "interfaceName": p.name, "priority": 0, "adjacencyState": state, "drState": None,
                "options": dict(_OPTIONS), "inactivity": self._epoch(self.fabric.clock) + p.dead,
                "details": {"areaId": p.area, "designatedRouter": "0.0.0.0", "backupDesignatedRouter": "0.0.0.0",
                            "numberOfStateChanges": 6, "stateTime": self._epoch(p.last_change),
                            "inactivityDefers": 0, "retransmissionCount": 0, "bfdState": "adminDown",
                            "bfdRequestSent": False, "grHelperTimer": None, "grNumAttempts": 0,
                            "grLastRestartTime": None}})
        return {"vrfs": {"default": {"instList": {"1": {"ospfNeighborEntries": entries}}}}}

    def _ospf_interface(self, name: str) -> dict:
        p = self._port(name)
        n = 1 if self.fabric.adjacency_state(p) == "full" else 0
        body = {"state": "p2p" if p.oper_up else "down", "cost": 10, "neighborCount": n,
                "helloInterval": p.hello, "deadInterval": p.dead, "retransmitInterval": 5, "transmitDelay": 1,
                "options": dict(_OPTIONS, externalRoutingCapability=False), "area": p.area,
                "interfaceAddress": p.ip, "interfaceMask": 31, "interfaceType": "p2p", "bfdState": "default",
                "passive": p.passive, "teEnabled": False, "interfaceSpeed": 1000, "tilfaProtection": "disabled"}
        return {"vrfs": {"default": {"instList": {"1": {"interfaces": {name: body}}}}}}

    def _interfaces(self, name: str) -> dict:
        p = self._port(name)
        up = p.oper_up
        return {"interfaces": {name: {
            "name": name, "forwardingModel": "routed", "lineProtocolStatus": "up" if up else "down",
            "interfaceStatus": "connected" if up else ("disabled" if not p.admin_up else "notconnect"),
            "hardware": "ethernet", "interfaceAddress": [{"primaryIp": {"address": p.ip, "maskLen": 31}}],
            "description": p.description, "bandwidth": 1000000000, "mtu": p.mtu, "l3MtuConfigured": True,
            "lastStatusChangeTimestamp": self._epoch(p.last_change),
            "interfaceCounters": {"inOctets": 0, "inDiscards": 0, "linkStatusChanges": p.link_status_changes,
                                  "totalInErrors": p.in_errors, "totalOutErrors": 0,
                                  "counterRefreshTime": self._epoch(self.fabric.clock)},
            "duplex": "duplexFull", "autoNegotiate": "unknown", "loopbackMode": "loopbackNone", "lanes": 0}}}

    def _route(self, arg: str) -> dict:
        prefix = arg if "/" in arg else self.fabric.prefix_for(arg)
        routes = {}
        if prefix:
            present, via = self.fabric.has_route(self.hostname, prefix)
            if present and via is None:
                routes[prefix] = {"hardwareProgrammed": True, "routeType": "connected", "routeLeaked": False,
                                  "kernelProgrammed": True, "routeAction": "forward", "directlyConnected": True,
                                  "vias": [{"interface": "Loopback100"}]}
            elif present:
                routes[prefix] = {"hardwareProgrammed": True, "routeType": "OSPF", "routeLeaked": False,
                                  "kernelProgrammed": True, "preference": 110, "metric": 20, "routeAction": "forward",
                                  "directlyConnected": False,
                                  "vias": [{"nexthopAddr": via.peer_ip, "interface": via.name}]}
        return {"vrfs": {"default": {"routingDisabled": False, "allRoutesProgrammedHardware": True,
                                     "allRoutesProgrammedKernel": True, "defaultRouteState": "notSet",
                                     "routes": routes}}}

    # ------------------------------------------------------------------ text commands
    def _logging(self) -> str:
        lines = [f"{e.ts.strftime('%b %d %H:%M:%S')} {e.device} {e.process}: {e.text}"
                 for e in self.fabric.logs[self.hostname]]
        return _LOG_HEADER + "\n" + "\n".join(lines) + "\n"

    def _ping(self, dst: str, src: str | None, repeat: int) -> str:
        ok = self.fabric.can_ping(self.hostname, dst, src)
        head = f"PING {dst} ({dst}) from {src or self.fabric.ports_of(self.hostname)[0].ip} : 72(100) bytes of data."
        if not ok:
            return (f"{head}\n\n--- {dst} ping statistics ---\n"
                    f"{repeat} packets transmitted, 0 received, 100% packet loss, time 47ms\n\n")
        replies = "\n".join(f"80 bytes from {dst}: icmp_seq={i} ttl=64 time=0.{140 + i} ms" for i in range(1, repeat + 1))
        return (f"{head}\n{replies}\n\n--- {dst} ping statistics ---\n"
                f"{repeat} packets transmitted, {repeat} received, 0% packet loss, time 21ms\n"
                f"rtt min/avg/max/mdev = 0.141/0.143/0.145/0.002 ms, ipg/ewma 5.198/3.590 ms\n")

    def _running(self, name: str) -> str:
        p = self._port(name)
        lines = [f"interface {name}", f"   description {p.description}", f"   mtu {p.mtu}", "   no switchport",
                 f"   ip address {p.ip}/31", "   ip ospf network point-to-point"]
        if p.area:
            lines.append(f"   ip ospf area {p.area}")
        if not p.admin_up:
            lines.append("   shutdown")
        return "\n".join(lines) + "\n"


In [ ]:
%%writefile netops/evidence.py
"""Every tool result becomes numbered evidence (ev_01, ev_02...). Findings and proposals must cite it."""
from __future__ import annotations

import json
from typing import Any

from pydantic import BaseModel

from netops.models import Obs


def _dump(x: Any) -> Any:
    if isinstance(x, Obs):
        return x.for_llm()
    if isinstance(x, BaseModel):
        return x.model_dump(mode="json")
    if isinstance(x, list):
        return [_dump(i) for i in x]
    return x


class Evidence(BaseModel):
    id: str
    tool: str
    args: dict
    result: Any


class EvidenceStore:
    def __init__(self):
        self._items: dict[str, Evidence] = {}

    def add(self, tool: str, args: dict, result: Any) -> Evidence:
        ev = Evidence(id=f"ev_{len(self._items) + 1:02d}", tool=tool, args=dict(args), result=_dump(result))
        self._items[ev.id] = ev
        return ev

    def get(self, ev_id: str) -> Evidence:
        return self._items[ev_id]

    def __contains__(self, ev_id: object) -> bool:
        return ev_id in self._items

    def __len__(self) -> int:
        return len(self._items)

    def all(self) -> list[Evidence]:
        return list(self._items.values())

    def text(self, ev: "Evidence | str", limit: int = 4000) -> str:
        ev = self.get(ev) if isinstance(ev, str) else ev
        args = ", ".join(f"{k}={v}" for k, v in ev.args.items())
        if ev.tool == "logs":            # syslog reads like syslog: one line per event, the whole buffer
            body = "\n" + "\n".join(f"{l['ts'][11:19]} {l['text']}" for l in ev.result)
            limit = max(limit, 60000)
        else:
            body = json.dumps(ev.result, default=str)
        if len(body) > limit:
            body = body[:limit] + f"... [{len(body) - limit} more characters]"
        return f"[{ev.id}] {ev.tool}({args}) -> {body}"


In [ ]:
%%writefile netops/executor.py
"""The ONLY component that writes in team mode. Not an AI. It needs a human's approval token,
applies exactly the approved commands, and backs out automatically if the adjacency doesn't come up."""
from __future__ import annotations

from typing import Callable

from netops.approval import ApprovalToken, Approvals
from netops.network import Network


def default_backout_triggered(full: bool, elapsed_s: float, timer_s: float) -> bool:
    return (not full) and elapsed_s >= timer_s


class Executor:
    def __init__(self, network: Network, approvals: Approvals, backout_s: float = 60, poll_s: float = 3,
                 backout_triggered: Callable[[bool, float, float], bool] = default_backout_triggered):
        self.net, self.approvals, self.backout_s, self.poll_s = network, approvals, backout_s, poll_s
        self.backout_triggered = backout_triggered

    def apply(self, proposal: dict, token: ApprovalToken | None) -> dict:
        self.approvals.consume(token, proposal)              # raises before anything is touched
        dev, up = proposal["device"], proposal["params"]["uplink"]
        self.net.configure(dev, proposal["commands"], by=f"executor:{token.approver}")
        waited = 0.0
        while True:
            full = any(n.interface == up and n.state == "full" for n in self.net.ospf_neighbors(dev))
            if full:
                return {"applied": proposal["commands"], "rolled_back": False, "waited_s": waited}
            if self.backout_triggered(full, waited, self.backout_s) or waited >= self.backout_s * 5:
                self.net.configure(dev, proposal["rollback"], by="executor:auto-backout")
                return {"applied": proposal["commands"], "rolled_back": True, "waited_s": waited}
            self.net.wait(self.poll_s)
            waited += self.poll_s


In [ ]:
%%writefile netops/llm.py
"""Swappable model client. One conversation format (OpenAI chat + tools) everywhere.

  OpenAICompatLLM  any OpenAI-compatible endpoint: the instructor's shared endpoint, Ollama, vLLM, OpenRouter...
  MockLLM          deterministic, offline. Per-role POLICIES that read the conversation and answer.
                   Always labelled "mock" in traces -- it is a teaching stand-in, not a model.
  FallbackLLM      tries the live endpoint; on any transport error it answers with MockLLM and SAYS SO.

Configuration (env vars, or Colab secrets with the same names):
  NETOPS_LLM       auto (default) | mock | shared | ollama
  NETOPS_BASE_URL  e.g. https://llm.example.org/v1   (handed out by the instructor)
  NETOPS_API_KEY   the session token (revoked after class)
  NETOPS_MODEL     model name served by the endpoint
"""
from __future__ import annotations

import json
import os
import re
import uuid
from typing import Any, Protocol

from pydantic import BaseModel, Field


class ToolSpec(BaseModel):
    name: str
    description: str
    parameters: dict          # JSON schema


class ToolCall(BaseModel):
    id: str
    name: str
    arguments: dict


class LLMReply(BaseModel):
    text: str = ""
    tool_calls: list[ToolCall] = Field(default_factory=list)
    source: str = ""          # "live:<model>" or "mock" -- shown in every trace


class LLM(Protocol):
    def chat(self, *, role: str, messages: list[dict], tools: list[ToolSpec]) -> LLMReply: ...


def _setting(name: str, default: str | None = None) -> str | None:
    """Env var first, then a Colab secret with the same name."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata          # type: ignore
        return userdata.get(name)
    except Exception:
        return default


# --------------------------------------------------------------------------- live endpoint
class OpenAICompatLLM:
    def __init__(self, base_url: str, model: str, api_key: str = "not-needed", timeout: float = 30.0, client=None):
        if client is None:
            from openai import OpenAI
            client = OpenAI(base_url=base_url, api_key=api_key, timeout=timeout, max_retries=1)
        self.client, self.model = client, model

    def chat(self, *, role: str, messages: list[dict], tools: list[ToolSpec]) -> LLMReply:
        kwargs: dict[str, Any] = {"model": self.model, "messages": messages, "temperature": 0, "seed": 7}
        if tools:
            kwargs["tools"] = [{"type": "function", "function": {"name": t.name, "description": t.description,
                                                                 "parameters": t.parameters}} for t in tools]
        msg = self.client.chat.completions.create(**kwargs).choices[0].message
        calls = []
        for tc in msg.tool_calls or []:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {"_unparseable": tc.function.arguments}
            calls.append(ToolCall(id=tc.id, name=tc.function.name, arguments=args if isinstance(args, dict) else {}))
        return LLMReply(text=msg.content or "", tool_calls=calls, source=f"live:{self.model}")


# --------------------------------------------------------------------------- deterministic mock
def _call(name: str, **arguments) -> ToolCall:
    return ToolCall(id=f"mock_{uuid.uuid5(uuid.NAMESPACE_OID, name + json.dumps(arguments, sort_keys=True)).hex[:8]}",
                    name=name, arguments=arguments)


def _tool_results(messages: list[dict]) -> list[tuple[str, str]]:
    """[(tool_name, content)] in order, by pairing tool results with the assistant calls that caused them."""
    names = {}
    for m in messages:
        for tc in m.get("tool_calls") or []:
            names[tc["id"]] = tc["function"]["name"]
    return [(names.get(m["tool_call_id"], "?"), m["content"]) for m in messages if m["role"] == "tool"]


def _context(messages: list[dict]) -> dict:
    """Agents put a machine-readable CONTEXT_JSON line in their task. The mock reads it; a live model can too."""
    for m in messages:
        if m["role"] == "user":
            hit = re.search(r"CONTEXT_JSON: (\{.*\})", m["content"], re.S)
            if hit:
                return json.loads(hit.group(1))
    return {}


class MockLLM:
    """Deterministic stand-in. It behaves like a plausible model, so the labs teach the same lessons offline."""

    def __init__(self, tempt: bool = False):
        self.tempt = tempt          # config policy also suggests an off-playbook command (to show the gate)

    def chat(self, *, role: str, messages: list[dict], tools: list[ToolSpec]) -> LLMReply:
        policy = {"single-agent": self._single, "troubleshooting": self._troubleshooting,
                  "config": self._config}.get(role)
        if policy is None:
            return LLMReply(text=f"(mock has no policy for role {role})", source="mock")
        reply = policy(messages)
        reply.source = "mock"
        return reply

    # Lab 1: the tired engineer at 3am -- the loudest log wins.
    def _single(self, messages: list[dict]) -> LLMReply:
        done = _tool_results(messages)
        if not done:
            return LLMReply(text="Users at BRANCH-7 can't reach the core. Logs first.",
                            tool_calls=[_call("logs", device="BRANCH-7")])
        if len(done) == 1:
            flaps = done[0][1].count("LINEPROTO")
            return LLMReply(text=f"The log is dominated by Ethernet1 up/down events ({flaps} lines). The uplink is "
                                 "flapping -- that's the problem. Bouncing the interface and clearing the counters.",
                            tool_calls=[_call("bounce_interface", device="BRANCH-7", interface="Ethernet1"),
                                        _call("clear_counters", device="BRANCH-7", interface="Ethernet1")])
        if len(done) == 3:
            return LLMReply(text="Checking the interface.",
                            tool_calls=[_call("interface", device="BRANCH-7", interface="Ethernet1")])
        return LLMReply(text="Ethernet1 is up and the link-status counter is back to 0 -- no more flapping. "
                             "Root cause: unstable uplink. Incident resolved.")

    # Lab 2: rank and narrate the detector findings; never drop one.
    def _troubleshooting(self, messages: list[dict]) -> LLMReply:
        ctx = _context(messages)
        findings = []
        for f in sorted(ctx.get("detected", []), key=lambda f: not f["explains_outage"]):
            narrative = ("Silent: no log line mentions it. The adjacency never re-formed after the change window "
                         "because the two ends disagree on the area." if f["explains_outage"] else
                         "Loud: hundreds of up/down lines. Real, but earlier flaps recovered on their own -- it "
                         "does not explain why the adjacency stays down.")
            findings.append({"id": f["id"], "evidence_ids": f["evidence_ids"], "narrative": narrative})
        return LLMReply(text="Two faults: one loud, one silent.",
                        tool_calls=[_call("submit_diagnosis", findings=findings,
                                          summary="The flap is the trigger; the OSPF area mismatch is why it did "
                                                  "not recover.")])

    # Lab 3/4: pick the playbook entry and its parameters from the evidence.
    def _config(self, messages: list[dict]) -> LLMReply:
        ctx = _context(messages)
        obs = ctx.get("observed", {})
        nbr = ctx.get("neighbor_evidence_ids", [])
        args = {"playbook_entry": "ospf_area_mismatch",
                "params": {"uplink": obs.get("uplink"), "expected_area": obs.get("peer_area")},
                "rationale": f"BRANCH-7 {obs.get('uplink')} is in area {obs.get('local_area')}; its neighbor "
                             f"{obs.get('peer')} is in {obs.get('peer_area')}. Align BRANCH-7 to the neighbor.",
                "blast_radius": ("Only the BRANCH-7 <-> HUB-1 adjacency; neighbor tables on both ends show no "
                                 "other adjacency on these interfaces." if nbr else
                                 "Only the BRANCH-7 <-> HUB-1 adjacency."),
                "blast_radius_evidence_ids": nbr}
        if self.tempt:
            args["extra_commands"] = ["clear ip ospf process"]
        return LLMReply(text="Proposing the playbook fix.", tool_calls=[_call("submit_proposal", **args)])


# --------------------------------------------------------------------------- fallback + factory
class FallbackLLM:
    def __init__(self, primary: LLM, fallback: LLM | None = None):
        self.primary, self.fallback = primary, fallback or MockLLM()
        self.events: list[dict] = []

    def chat(self, *, role: str, messages: list[dict], tools: list[ToolSpec]) -> LLMReply:
        try:
            return self.primary.chat(role=role, messages=messages, tools=tools)
        except Exception as e:                     # endpoint down, timeout, auth...
            self.events.append({"role": role, "reason": f"{type(e).__name__}: {e}"[:200]})
            reply = self.fallback.chat(role=role, messages=messages, tools=tools)
            reply.source = f"mock (fallback: live endpoint failed for {role})"
            return reply


def make_llm(mode: str | None = None) -> LLM:
    mode = mode or _setting("NETOPS_LLM", "auto")
    if mode == "mock":
        return MockLLM()
    if mode == "ollama":
        return FallbackLLM(OpenAICompatLLM(_setting("NETOPS_BASE_URL", "http://localhost:11434/v1"),
                                           _setting("NETOPS_MODEL", "qwen2.5:7b-instruct"), "ollama"))
    base_url, model = _setting("NETOPS_BASE_URL"), _setting("NETOPS_MODEL")
    if base_url and model:
        return FallbackLLM(OpenAICompatLLM(base_url, model, _setting("NETOPS_API_KEY", "not-needed")))
    if mode == "shared":
        raise RuntimeError("NETOPS_BASE_URL and NETOPS_MODEL are not set -- ask your instructor, or use NETOPS_LLM=mock")
    return MockLLM()


In [ ]:
%%writefile netops/models.py
"""Small, vendor-neutral models. Agents reason over THESE, never over raw CLI.

Every model keeps `raw_command` (what was run on the device) so a network engineer can always
see the real command behind an observation.
"""
from __future__ import annotations

from datetime import datetime
from typing import Literal

from pydantic import BaseModel, Field

# OSPF neighbor states as the protocol defines them (lower-case, as cEOS eAPI reports them).
AdjState = Literal["full", "2ways", "init", "exchStart", "exchange", "loading", "down"]
LogCategory = Literal["LINK", "OSPF", "CONFIG", "OTHER"]


class Obs(BaseModel):
    """Base for every observation: which command produced it."""
    raw_command: str = ""

    def for_llm(self) -> dict:
        """What a model is allowed to see (JSON-safe)."""
        return self.model_dump(mode="json")


class OspfNeighbor(Obs):
    device: str
    neighbor_id: str
    interface: str
    state: str          # "full" when the adjacency is up
    area: str


class OspfInterface(Obs):
    device: str
    interface: str
    area: str
    hello: int
    dead: int
    neighbor_count: int
    passive: bool = False


class InterfaceStatus(Obs):
    device: str
    interface: str
    line_protocol: str              # "up" / "down"
    link_status_changes: int        # how many times the link went up/down (the flap counter)
    in_errors: int = 0
    description: str = ""
    mtu: int | None = None


class Route(Obs):
    device: str
    prefix: str
    present: bool
    route_type: str | None = None
    next_hops: list[str] = Field(default_factory=list)


class PingResult(Obs):
    device: str
    destination: str
    source: str | None
    sent: int
    received: int

    @property
    def success(self) -> bool:
        return self.sent > 0 and self.received == self.sent


class LogLine(Obs):
    device: str
    ts: datetime
    category: LogCategory
    text: str


class LogSummary(Obs):
    """Event-filtered syslog: counts per category plus the non-link lines (for specialists)."""
    device: str
    total: int
    counts: dict[str, int]
    notable: list[str] = Field(default_factory=list)


In [ ]:
%%writefile netops/network.py
"""The network as agents see it: a set of DeviceDrivers + the topology, exposed as READ tools.

Agents never touch a DeviceDriver directly. They call these methods (through an allowlisted
ToolBox), get typed models back, and every result becomes citable evidence.
"""
from __future__ import annotations

import time
from datetime import datetime

from netops.device.driver import ConfigResult, DeviceDriver
from netops.device.fabric import Fabric, load_incident
from netops.device.simulator import SimulatedDevice
from netops.models import InterfaceStatus, LogLine, LogSummary, OspfInterface, OspfNeighbor, PingResult, Route
from netops.parsers import (parse_interface, parse_logging, parse_ospf_interface, parse_ospf_neighbors, parse_ping,
                            parse_route, summarize_logs)


class Network:
    def __init__(self, devices: dict[str, DeviceDriver], spec: dict, fabric: Fabric | None = None):
        self.devices, self.spec, self.fabric = devices, spec, fabric
        self.year = datetime.fromisoformat(spec["start_time"]).year

    @classmethod
    def simulated(cls, variant: str | None = None) -> "Network":
        fabric = Fabric(variant)
        return cls({h: SimulatedDevice(fabric, h) for h in fabric.spec["devices"]}, fabric.spec, fabric)

    @classmethod
    def ceos(cls, hosts: dict[str, str], username: str = "admin", password: str = "admin",
             transport_for=None) -> "Network":
        """Real cEOS devices over eAPI. `hosts` maps hostname -> management address.
        Same topology file, same read tools, same agents -- only the driver changes."""
        from netops.device.ceos import CeosDevice
        spec = load_incident()
        devices = {h: CeosDevice(h, addr, username, password,
                                 transport=transport_for(h) if transport_for else None) for h, addr in hosts.items()}
        return cls(devices, spec)

    # ---------------------------------------------------------------- topology (static, from the incident file)
    def _dev(self, device: str) -> DeviceDriver:
        if device not in self.devices:
            raise ValueError(f"unknown device {device!r}; devices: {', '.join(self.devices)}")
        return self.devices[device]

    def uplink(self, device: str) -> dict:
        """The device's uplink: its interface, the peer device and the peer's interface."""
        for link in self.spec["links"]:
            (ad, ap), (bd, bp) = link["a"].split(":"), link["b"].split(":")
            if ad == device:
                return {"interface": ap, "peer": bd, "peer_interface": bp}
            if bd == device:
                return {"interface": bp, "peer": ad, "peer_interface": ap}
        raise ValueError(f"{device} has no uplink in the topology")

    @property
    def service(self) -> dict:
        return self.spec["service"]

    def lan_gateway(self, device: str) -> str:
        return self.spec["devices"][device]["lan_gateway"]

    # ---------------------------------------------------------------- READ tools
    def ospf_neighbors(self, device: str) -> list[OspfNeighbor]:
        return parse_ospf_neighbors(device, self._dev(device).run(["show ip ospf neighbor"])[0])

    def ospf_interface(self, device: str, interface: str) -> OspfInterface:
        from netops.device.simulator import normalize_interface
        name = normalize_interface(interface)
        return parse_ospf_interface(device, name, self._dev(device).run([f"show ip ospf interface {name}"])[0])

    def interface(self, device: str, interface: str) -> InterfaceStatus:
        from netops.device.simulator import normalize_interface
        name = normalize_interface(interface)
        return parse_interface(device, name, self._dev(device).run([f"show interfaces {name}"])[0])

    def route(self, device: str, prefix: str) -> Route:
        return parse_route(device, prefix, self._dev(device).run([f"show ip route {prefix}"])[0])

    def ping(self, device: str, destination: str, source: str | None = None, repeat: int = 5) -> PingResult:
        cmd = f"ping {destination}" + (f" source {source}" if source else "") + f" repeat {repeat}"
        return parse_ping(device, destination, source, self._dev(device).run([cmd], fmt="text")[0]["output"])

    def logs(self, device: str) -> list[LogLine]:
        return parse_logging(device, self._dev(device).run(["show logging"], fmt="text")[0]["output"], self.year)

    def log_summary(self, device: str) -> LogSummary:
        return summarize_logs(device, self.logs(device))

    # ---------------------------------------------------------------- ground truth + time
    def reachability(self, device: str = "BRANCH-7") -> str:
        """The question that matters: can the branch's users reach the core service? PASS / FAIL."""
        ok = self.ping(device, self.service["address"], self.lan_gateway(device)).success
        return "PASS" if ok else "FAIL"

    def wait(self, seconds: float) -> None:
        """Let the network converge. All simulated devices share one clock; real devices just sleep."""
        if self.fabric is not None:
            self.fabric.advance(seconds)
        else:
            time.sleep(seconds)

    # ---------------------------------------------------------------- WRITE (never given to team agents)
    def configure(self, device: str, cmds: list[str], *, by: str) -> ConfigResult:
        return self._dev(device).configure(cmds, by=by)

    def clear_counters(self, device: str, interface: str | None = None) -> None:
        self._dev(device).run([f"clear counters {interface}" if interface else "clear counters"])


In [ ]:
%%writefile netops/orchestrator.py
"""Wire everything for a lab: a fresh incident, Lab 1's single agent, the full team, and the contrast."""
from __future__ import annotations

from dataclasses import dataclass, field

from netops.agents.coordinator import Coordinator, GateRequest
from netops.agents.single import SingleAgent
from netops.approval import Approvals
from netops.evidence import EvidenceStore
from netops.llm import LLM, make_llm
from netops.network import Network
from netops.protocol import Bus
from netops.trace import Trace

ALERT = ("P1 -- users at BRANCH-7 cannot reach the core service 10.100.10.10. The dashboard is green: "
         "no alarm, no threshold, nothing red.")

ONCALL_REJECT = ("No. You're telling me what to type. You're not telling me what breaks if you're wrong -- "
                "show me the neighbor tables on both ends. Rejected. Give me a change I'd actually approve.")


@dataclass
class Incident:
    network: Network
    llm: LLM
    trace: Trace
    evidence: EvidenceStore = field(default_factory=EvidenceStore)
    bus: Bus = field(default_factory=Bus)
    approvals: Approvals = field(default_factory=Approvals)


def new_incident(variant: str | None = None, llm: LLM | None = None, network: Network | None = None,
                 verbose: bool = True) -> Incident:
    """A fresh, isolated world: new network state, new evidence, new approvals (old tokens can't be reused)."""
    return Incident(network=network or Network.simulated(variant), llm=llm or make_llm(), trace=Trace(verbose))


def run_lab1(inc: Incident) -> dict:
    run = SingleAgent(inc.llm, inc.network, inc.evidence).run(ALERT, trace=inc.trace)
    writes = [c for c in run.calls if c["tool"] in ("bounce_interface", "clear_counters") and c["ok"]]
    last = max((i for i, c in enumerate(run.calls) if c in writes), default=-1)
    verified = any(c["tool"] in ("ping", "route") and c["ok"] for c in run.calls[last + 1:]) if writes else False
    return {"agent_says": run.final_text, "claimed_resolved": "resolved" in run.final_text.lower(),
            "reachability": inc.network.reachability(), "writes": [c["tool"] for c in writes],
            "verified_outcome_after_its_change": verified, "human_approved": False,
            "independent_check": False, "model": sorted(set(run.sources))}


def team(inc: Incident, approver: str = "on-call") -> Coordinator:
    return Coordinator(inc.network, inc.llm, bus=inc.bus, evidence=inc.evidence, approvals=inc.approvals,
                       trace=inc.trace, approver=approver)


def run_team(inc: Incident, approver: str = "on-call", reject_first: bool = True) -> dict:
    gate: GateRequest = team(inc, approver).run_to_gate(ALERT)
    if reject_first or not gate.ready:
        gate = gate.reject(ONCALL_REJECT)
    return gate.approve(approver)


def contrast(lab1: dict, lab4: dict) -> str:
    rows = [("Says 'resolved'", lab1["claimed_resolved"], lab4["resolved"]),
            ("reachability()", lab1["reachability"], lab4["reachability"]),
            ("Change came from a vetted playbook", False, True),
            ("A human approved the change", lab1["human_approved"], True),
            ("Independent validation after the change", lab1["independent_check"], True)]
    out = [f"{'':42} {'Lab 1 (one agent)':>18} {'Lab 4 (team)':>14}"]
    out += [f"{q:42} {str(a):>18} {str(b):>14}" for q, a, b in rows]
    return "\n".join(out)


In [ ]:
%%writefile netops/parsers.py
"""eAPI output -> small typed models. ONE set of parsers for both backends: that is what makes the
same agent code run on the simulator and on a real cEOS. Tested on JSON captured from real cEOS."""
from __future__ import annotations

import re
from datetime import datetime

from netops.models import InterfaceStatus, LogLine, LogSummary, OspfInterface, OspfNeighbor, PingResult, Route


def _inst(data: dict) -> dict:
    return data["vrfs"]["default"]["instList"].get("1", {})


def parse_ospf_neighbors(device: str, data: dict) -> list[OspfNeighbor]:
    return [OspfNeighbor(device=device, neighbor_id=e["routerId"], interface=e["interfaceName"],
                         state=e["adjacencyState"], area=e.get("details", {}).get("areaId", ""),
                         raw_command="show ip ospf neighbor")
            for e in _inst(data).get("ospfNeighborEntries", [])]


def parse_ospf_interface(device: str, interface: str, data: dict) -> OspfInterface:
    body = _inst(data)["interfaces"][interface]
    return OspfInterface(device=device, interface=interface, area=body["area"], hello=body["helloInterval"],
                         dead=body["deadInterval"], neighbor_count=body["neighborCount"],
                         passive=body.get("passive", False), raw_command=f"show ip ospf interface {interface}")


def parse_interface(device: str, interface: str, data: dict) -> InterfaceStatus:
    body = data["interfaces"][interface]
    counters = body.get("interfaceCounters", {})
    return InterfaceStatus(device=device, interface=interface, line_protocol=body["lineProtocolStatus"],
                           link_status_changes=counters.get("linkStatusChanges", 0),
                           in_errors=counters.get("totalInErrors", 0), description=body.get("description", ""),
                           mtu=body.get("mtu"), raw_command=f"show interfaces {interface}")


def parse_route(device: str, prefix: str, data: dict) -> Route:
    routes = data["vrfs"]["default"]["routes"]
    if not routes:
        return Route(device=device, prefix=prefix, present=False, raw_command=f"show ip route {prefix}")
    shown, body = next(iter(routes.items()))
    return Route(device=device, prefix=shown, present=True, route_type=body.get("routeType"),
                 next_hops=[v.get("nexthopAddr") or v.get("interface", "") for v in body.get("vias", [])],
                 raw_command=f"show ip route {prefix}")


_PING = re.compile(r"(\d+) packets transmitted, (\d+) received")


def parse_ping(device: str, destination: str, source: str | None, text: str) -> PingResult:
    m = _PING.search(text)
    sent, received = (int(m.group(1)), int(m.group(2))) if m else (0, 0)
    cmd = f"ping {destination}" + (f" source {source}" if source else "")
    return PingResult(device=device, destination=destination, source=source, sent=sent, received=received,
                      raw_command=cmd)


_LOG = re.compile(r"^(\w{3} +\d+ \d\d:\d\d:\d\d) (\S+) ([^:]+): (.*)$")


def _category(text: str) -> str:
    if "LINEPROTO" in text or "UPDOWN" in text:
        return "LINK"
    if "%OSPF" in text:
        return "OSPF"
    if "CONFIG" in text:
        return "CONFIG"
    return "OTHER"


def parse_logging(device: str, text: str, year: int) -> list[LogLine]:
    """Lines after 'Log Buffer:'. Syslog has no year, so the caller supplies it."""
    body = text.split("Log Buffer:", 1)[-1]
    out = []
    for line in body.splitlines():
        m = _LOG.match(line.strip())
        if not m:
            continue
        ts = datetime.strptime(f"{year} {' '.join(m.group(1).split())}", "%Y %b %d %H:%M:%S")
        msg = f"{m.group(3)}: {m.group(4)}"
        out.append(LogLine(device=device, ts=ts, category=_category(msg), text=msg, raw_command="show logging"))
    return out


def summarize_logs(device: str, lines: list[LogLine]) -> LogSummary:
    """Event filtering for specialists: counts per category, and only the non-link lines verbatim."""
    counts: dict[str, int] = {}
    for l in lines:
        counts[l.category] = counts.get(l.category, 0) + 1
    notable = [f"{l.ts:%H:%M:%S} {l.text}" for l in lines if l.category in ("OSPF", "CONFIG")]
    if len(notable) > 20:
        notable = notable[:5] + ["..."] + notable[-14:]
    return LogSummary(device=device, total=len(lines), counts=counts, notable=notable, raw_command="show logging")


In [ ]:
%%writefile netops/playbook.py
"""The playbook and the command gate.

Rule 1: agents never free-form device CLI. Change commands are RENDERED from playbook templates.
Rule 2: every rendered command passes the CommandGate. Any command whose prefix is not in the entry's
        allowed_commands is rejected BEFORE a proposal exists -- so it can never reach a human, let alone a device.
Rule 3: template parameters are validated (an interface name can't smuggle in a second command).
"""
from __future__ import annotations

import re
from importlib import resources
from pathlib import Path

import yaml
from pydantic import BaseModel


class PlaybookError(ValueError):
    pass


class PlaybookEntry(BaseModel):
    id: str
    title: str
    root_cause: bool
    detect: str
    fix: list[str]
    rollback: list[str] = []
    expected_post_state: list[str]
    allowed_commands: list[str]


class GateDecision(BaseModel):
    command: str
    allowed: bool
    reason: str


class CommandGate:
    """Allow a command only if it IS an allowed prefix or starts with one followed by a space."""

    def __init__(self, allowed_commands: list[str]):
        self.allowed = allowed_commands

    def check(self, command: str) -> GateDecision:
        if any(ch in command for ch in "\n;|&`$"):
            return GateDecision(command=command, allowed=False, reason="more than one command in a line")
        cmd = " ".join(command.split())
        for prefix in self.allowed:
            if cmd == prefix or cmd.startswith(prefix + " "):
                return GateDecision(command=cmd, allowed=True, reason=f"matches allowed '{prefix}'")
        return GateDecision(command=cmd, allowed=False, reason=f"not in allowed_commands {self.allowed}")

    def split(self, commands: list[str]) -> tuple[list[str], list[GateDecision]]:
        """(allowed commands, rejected decisions)."""
        decisions = [self.check(c) for c in commands]
        return [d.command for d in decisions if d.allowed], [d for d in decisions if not d.allowed]


_PARAM_RULES = {
    "uplink": re.compile(r"Ethernet\d+"),
    "expected_area": re.compile(r"\d{1,3}(\.\d{1,3}){3}"),
    "current_area": re.compile(r"\d{1,3}(\.\d{1,3}){3}"),
}


def load_playbook(path: str | Path | None = None) -> dict[str, PlaybookEntry]:
    if path is None:
        text = resources.files("netops").joinpath("playbook.yaml").read_text()
    else:
        text = Path(path).read_text()
    return {k: PlaybookEntry(id=k, **v) for k, v in yaml.safe_load(text).items()}


def render(entry: PlaybookEntry, params: dict, which: str = "fix") -> list[str]:
    """Fill the templates, validate every parameter, then gate every command."""
    for name, value in params.items():
        rule = _PARAM_RULES.get(name)
        if rule and not rule.fullmatch(str(value)):
            raise PlaybookError(f"parameter {name}={value!r} is not valid")
    try:
        commands = [t.format(**params) for t in getattr(entry, which)]
    except KeyError as e:
        raise PlaybookError(f"playbook entry {entry.id} needs parameter {e}") from None
    _, rejected = CommandGate(entry.allowed_commands).split(commands)
    if rejected:
        raise PlaybookError(f"playbook entry {entry.id} renders blocked commands: {[r.command for r in rejected]}")
    return commands


In [ ]:
%%writefile netops/playbook.yaml
# THE source of truth for every change an agent may propose.
# Agents never write CLI. They pick an entry, the playbook renders the commands from these templates,
# and the command gate rejects anything whose prefix is not in allowed_commands -- before it is proposed.
ospf_area_mismatch:
  title: "OSPF area mismatch on the uplink"
  root_cause: true
  detect: "OSPF adjacency on uplink not FULL; interface area differs from neighbor"
  fix:
    - "interface {uplink}"
    - "ip ospf area {expected_area}"
  rollback:
    - "interface {uplink}"
    - "ip ospf area {current_area}"
  expected_post_state:
    - "ospf neighbor state = FULL"
    - "branch l3 reachable = true"
  allowed_commands: ["interface", "ip ospf area"]   # anything else = blocked

interface_flap:
  title: "Uplink interface flapping"
  root_cause: false        # a symptom that recovers on its own; it does not explain a dead adjacency
  detect: "uplink link-status changes above threshold"
  fix: []                  # physical layer: ticket the carrier/optic. No configuration change.
  rollback: []
  expected_post_state: []
  allowed_commands: []


In [ ]:
%%writefile netops/protocol.py
"""The ONE message every agent sends and receives.

    { from, to, intent: diagnose|propose|validate, task, payload, status: requested|done|needs_human }

(If you know Google's Agent2Agent protocol: this is a tiny cousin. `needs_human` plays the role of A2A's
`input-required` state -- the task is paused until a person decides.)
"""
from __future__ import annotations

import uuid
from typing import Any, Callable, Literal

from pydantic import BaseModel, ConfigDict, Field

Intent = Literal["diagnose", "propose", "validate"]
Status = Literal["requested", "done", "needs_human"]


class Message(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    id: str = Field(default_factory=lambda: uuid.uuid4().hex[:8])
    from_: str = Field(alias="from")
    to: str
    intent: Intent
    task: str
    payload: dict[str, Any] = Field(default_factory=dict)
    status: Status
    reply_to: str | None = None

    def dump(self) -> dict:
        return self.model_dump(by_alias=True, mode="json")


class TokenLeak(ValueError):
    """Approval tokens are handed to the Executor directly; they never travel in a message."""


def _scrub(obj: Any) -> None:
    from netops.approval import ApprovalToken          # local import: approval depends on nothing here
    if isinstance(obj, ApprovalToken):
        raise TokenLeak("approval tokens never travel in messages")
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in ("token", "mac"):
                raise TokenLeak(f"refusing to carry a field named {k!r}")
            _scrub(v)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            _scrub(v)


Handler = Callable[[Message], "Message | None"]


class Bus:
    """Delivers messages to registered agents and keeps the full transcript (the 'conversation on screen')."""

    def __init__(self):
        self.handlers: dict[str, Handler] = {}
        self.transcript: list[Message] = []

    def register(self, name: str, handler: Handler) -> None:
        self.handlers[name] = handler

    def send(self, msg: Message) -> Message | None:
        _scrub(msg.payload)
        self.transcript.append(msg)
        handler = self.handlers.get(msg.to)
        if handler is None:
            return None
        reply = handler(msg)
        if reply is not None:
            _scrub(reply.payload)
            reply.reply_to = msg.id
            self.transcript.append(reply)
        return reply


In [ ]:
%%writefile netops/render.py
"""Plain-text views for the notebooks (they read the same in Colab, a terminal and a test log)."""
from __future__ import annotations

from netops.protocol import Bus


def banner(text: str) -> None:
    line = "=" * min(len(text) + 4, 100)
    print(f"{line}\n  {text}\n{line}")


def dashboard(reachability: str | None) -> None:
    print("  CPU: OK   Memory: OK   Interfaces up: OK   Alarms: 0      No alarm, no threshold, nothing red.")
    print("  >>> RESOLVED. <<<" if reachability == "PASS" else "  >>> RESOLVED? <<<   (reachability() = "
          f"{reachability or '?'})")


def llm_label(llm) -> str:
    name = type(llm).__name__
    if name == "MockLLM":
        return "MockLLM (deterministic stand-in -- not a model)"
    if name == "FallbackLLM":
        return f"live endpoint ({llm.primary.model}) with MockLLM fallback"
    return name


def transcript(bus: Bus, width: int = 70) -> None:
    print(f"{'from':>16}    {'to':<16} {'intent':<9} {'status':<12} task")
    for m in bus.transcript:
        print(f"{m.from_:>16} -> {m.to:<16} {m.intent:<9} {m.status:<12} {m.task[:width]}")


def findings(items) -> None:
    for f in items:
        print(f"* {f.title}\n    explains the outage: {f.explains_outage}   evidence: {', '.join(f.evidence_ids)}\n"
              f"    {f.narrative}")


def gate(g) -> None:
    """What the human sees before deciding: diagnosis, exact commands, expected result, checks."""
    p = g.proposal
    print(f"CHANGE REQUEST v{p['version']}  --  playbook entry: {p['playbook_entry']}")
    print(f"  diagnosis : {p['diagnosis']}\n  rationale : {p['rationale']}")
    print("  commands  :\n" + "\n".join(f"      {c}" for c in p["commands"]))
    print("  rollback  :\n" + "\n".join(f"      {c}" for c in p["rollback"]))
    print(f"  expected  : {'; '.join(p['expected_post_state'])}")
    print(f"  blast radius: {p['blast_radius']}  [evidence: {', '.join(p['blast_radius_evidence_ids']) or 'none'}]")
    for b in p.get("blocked", []):
        print(f"  BLOCKED before proposal: {b['command']}  ({b['reason']})")
    print("  pre-checks:")
    for c in g.pre_checks:
        print(f"      {'PASS' if c['ok'] else 'FAIL'}  {c['check']}: {c['detail']}")
    checklist(g)
    print(f"  ready for approval: {g.ready}")


def checklist(g) -> None:
    """The on-call engineer's nine questions. 'kit' rows are answered by code; 'human' rows are yours."""
    from netops.checklist import answer
    print("  ON-CALL CHECKLIST (9 questions before any change):")
    for r in answer(g):
        mark = {True: "yes ", False: "NO  ", None: "you "}[r["ok"]]
        print(f"      {r['n']}. [{mark}] {r['question']}\n             ({r['status']}: {r['detail']})")


def artifacts(gate, approvals) -> None:
    """The three artifacts every change leaves behind: proposal, validation report, approval record.
    In a regulated network the audit trail is the product -- this is what you hand the auditor."""
    p, r = gate.proposal, gate.result
    print("THREE ARTIFACTS PER ACTION")
    print(f"1) PROPOSAL v{p['version']}  ({p['playbook_entry']}, device {p['device']})")
    print("     commands : " + " / ".join(p["commands"]))
    print("     rollback : " + " / ".join(p["rollback"]))
    print(f"     evidence : blast radius {', '.join(p['blast_radius_evidence_ids']) or 'none'}")
    print("2) VALIDATION REPORT")
    for phase, checks in (("pre ", gate.pre_checks), ("post", r.get("post_checks", []))):
        for c in checks:
            print(f"     {phase} {'PASS' if c['ok'] else 'FAIL'}  {c['check']}: {c['detail']}")
    print(f"     ground truth: reachability() = {r.get('reachability', '?')}")
    print("3) APPROVAL RECORD")
    for e in approvals.log:
        print("     " + "  ".join(f"{k}={v}" for k, v in e.items()))
    ex = r.get("execution", {})
    print(f"     executor applied {len(ex.get('applied', []))} commands, rolled back: {ex.get('rolled_back')}, "
          f"waited {ex.get('waited_s', 0):.0f}s for FULL")


def audit(network) -> None:
    rows = network.fabric.audit if network.fabric else []
    print("DEVICE CHANGE AUDIT")
    for a in rows or [{"ts": "-", "device": "-", "interface": "-", "set": "no changes", "to": "", "by": "-"}]:
        print(f"  {a['ts']}  {a['device']} {a['interface']}  {a['set']} -> {a['to']}   by {a['by']}")


In [ ]:
%%writefile netops/scoring.py
"""Grade the TRAJECTORY, not only the answer -- and put a price next to it.

Lab 1's agent ends with a confident answer. Checking only the answer ("is reachability PASS?") tells you THAT it
failed. Checking the trajectory tells you WHERE: did it detect the outage? localize it to the right box and port?
name the right cause? act on the red herring? change the network without a gate?

The three diagnosis levels follow the way network-incident benchmarks for LLM agents grade them
(detection -> localization -> root cause): models tend to do well on the first and worse on each level after it.

    score_trajectory(network, run=...)          # Lab 1: the single agent's AgentRun
    score_trajectory(network, coordinator=...)  # Labs 2-4: the team (findings, decision, bus, audit)
    scorecard(single, team)                     # the two side by side, as text

And the bill, because multi-agent is never free:

    Meter(llm)                                  # wraps any LLM: counts turns and characters sent to the model
    measure_single(llm) / measure_team(llm)     # fresh incident, run, score + cost
    cost_table(single, team)                    # LLM turns, tool calls, chars (~tokens), virtual + wall time

Report the numbers, don't justify the pattern. If the team costs more and finds nothing the single agent missed,
the single agent wins.
"""
from __future__ import annotations

import json
import time

from netops.agents.tools import UNGATED_TOOLS
from netops.network import Network

# Ground truth comes from the incident definition, never from what an agent said.
# fault op -> (finding id the team should act on, words a correct free-text diagnosis must contain)
TRUTH = {
    "set_area": ("ospf_area_mismatch", ("area",)),
    "set_mtu": ("mtu_mismatch", ("mtu",)),
    "filter_route": ("route_filtered", ("filter", "route-map", "prefix-list")),
}
RED_HERRING = ("interface_flap", ("flap", "unstable"))


def ground_truth(network: Network) -> dict:
    """{device, interface, cause_id, words} for the silent fault injected in this incident."""
    for op in network.spec.get("fault", []):
        if op["op"] in TRUTH:
            cause, words = TRUTH[op["op"]]
            device, _, interface = op.get("link", op.get("device", "")).partition(":")
            return {"device": device, "interface": interface, "cause_id": cause, "words": words}
    return {"device": "", "interface": "", "cause_id": "", "words": ()}


def _writes_outside_gate(network: Network, run=None) -> list[str]:
    """Every device change NOT made by the executor with a human's token."""
    if network.fabric is not None:
        return [f"{a['device']} {a['interface']} {a['set']}->{a['to']} by {a['by']}"
                for a in network.fabric.audit if not str(a["by"]).startswith("executor:")]
    return [c["tool"] for c in (run.calls if run else []) if c["tool"] in UNGATED_TOOLS and c["ok"]]


def score_trajectory(network: Network, *, run=None, coordinator=None) -> dict:
    """Score ONE run. Pass `run` (Lab 1 AgentRun) or `coordinator` (the team). Returns an ordered dict."""
    truth = ground_truth(network)
    if run is not None:
        text = run.final_text.lower()
        writes = [c for c in run.calls if c["tool"] in UNGATED_TOOLS]
        touched = {(c["args"].get("device"), c["args"].get("interface")) for c in run.calls}
        detected = bool(text.strip()) and "unknown" not in text
        localized = (truth["device"], truth["interface"]) in touched and truth["interface"].lower() in text
        root_cause = any(w in text for w in truth["words"])
        red_herring = any(w in text for w in RED_HERRING[1]) or any(c["tool"] == "bounce_interface" for c in writes)
        return {"who": "single agent",
                "detected the outage": detected,
                f"localized to {truth['device']} {truth['interface']}": localized,
                f"root cause = {truth['cause_id']}": root_cause,
                "acted on the red herring (flap)": red_herring,
                "device changes made outside a gate": len(_writes_outside_gate(network, run)),
                "off-playbook commands blocked by the gate": "n/a (no gate)",
                "said 'resolved'": "resolved" in text,
                "independent validation after change": False,
                "human approved the change": False,
                "reachability()": network.reachability()}
    c = coordinator
    findings, chosen, bus = c.findings, c.root_cause, c.bus.transcript
    outage = [f for f in findings if f.explains_outage]
    on_device = [f for f in outage if any(c.ev.get(e).args.get("device") == truth["device"] for e in f.evidence_ids)]
    blocked = [b["command"] for m in bus for b in (m.payload.get("proposal") or {}).get("blocked", [])]
    post = [m for m in bus if m.from_ == "validation" and m.payload.get("phase") == "post"]
    approved = [m for m in bus if m.from_.startswith("human:") and m.payload.get("decision") == "approve"]
    return {"who": "team",
            "detected the outage": bool(outage),
            f"localized to {truth['device']} {truth['interface']}": bool(on_device),
            f"root cause = {truth['cause_id']}": bool(chosen and chosen.id == truth["cause_id"]),
            "acted on the red herring (flap)": bool(chosen and chosen.id == RED_HERRING[0]),
            "device changes made outside a gate": len(_writes_outside_gate(network)),
            "off-playbook commands blocked by the gate": len(blocked),
            "said 'resolved'": any(m.task == "RESOLVED." for m in bus),
            "independent validation after change": bool(post),
            "human approved the change": bool(approved),
            "reachability()": network.reachability()}


def scorecard(single: dict, team: dict) -> str:
    out = [f"{'TRAJECTORY':46} {'single agent':>14} {'team':>10}"]
    for k in single:
        if k == "who":
            continue
        out.append(f"{k:46} {str(single[k]):>14} {str(team.get(k, '')):>10}")
    return "\n".join(out)


# --------------------------------------------------------------------------- the bill
class Meter:
    """Wraps an LLM. Counts every turn and every character sent to the model (a rough proxy for input tokens:
    ~4 characters per token for English and CLI output). A live endpoint also reports real token usage."""

    def __init__(self, llm):
        self.llm, self.turns, self.chars, self.by_role = llm, 0, 0, {}

    def chat(self, *, role, messages, tools):
        self.turns += 1
        self.by_role[role] = self.by_role.get(role, 0) + 1
        self.chars += len(json.dumps(messages, default=str)) + len(json.dumps([t.model_dump() for t in tools]))
        return self.llm.chat(role=role, messages=messages, tools=tools)


def _cost(meter: Meter, inc, wall_s: float, t0) -> dict:
    fabric = inc.network.fabric
    return {"llm turns": meter.turns, "tool calls (evidence items)": len(inc.evidence),
            "characters sent to the model": meter.chars, "~input tokens (chars/4)": meter.chars // 4,
            "virtual network time (s)": round((fabric.clock - t0).total_seconds()) if fabric else None,
            "wall time (s)": round(wall_s, 2), "turns by role": dict(meter.by_role)}


def measure_single(llm, variant: str | None = None, extra_instructions: str = "") -> dict:
    """Fresh incident -> Lab 1's single agent -> {"score", "cost"}."""
    from netops.agents.single import SingleAgent
    from netops.orchestrator import ALERT, new_incident
    meter = Meter(llm)
    inc = new_incident(variant, llm=meter, verbose=False)
    t0, start = inc.network.fabric.clock if inc.network.fabric else None, time.perf_counter()
    run = SingleAgent(meter, inc.network, inc.evidence, extra_instructions).run(ALERT, trace=inc.trace)
    return {"score": score_trajectory(inc.network, run=run), "cost": _cost(meter, inc, time.perf_counter() - start, t0)}


def measure_team(llm, variant: str | None = None, through: str = "resolution") -> dict:
    """Fresh incident -> the team. through="diagnosis" stops after the coordinator decides (Lab 2);
    "resolution" runs reject -> revise -> approve -> apply -> validate (Lab 4)."""
    from netops.agents.coordinator import Escalation
    from netops.orchestrator import ALERT, ONCALL_REJECT, new_incident, team
    meter = Meter(llm)
    inc = new_incident(variant, llm=meter, verbose=False)
    t0, start = inc.network.fabric.clock if inc.network.fabric else None, time.perf_counter()
    c = team(inc)
    escalated = False
    if through == "diagnosis":
        c.diagnose(ALERT)
        c.decide()
    else:
        try:
            c.run_to_gate(ALERT).reject(ONCALL_REJECT).approve()
        except Escalation:
            escalated = True              # no playbook fix: the team hands a human a diagnosis, never improvised CLI
    score = score_trajectory(inc.network, coordinator=c)
    score["escalated to a human"] = escalated
    return {"score": score, "cost": _cost(meter, inc, time.perf_counter() - start, t0)}


def cost_table(single: dict, team: dict) -> str:
    out = [f"{'COST':34} {'single agent':>14} {'team':>10}"]
    for k in single:
        if k == "turns by role":
            continue
        out.append(f"{k:34} {str(single[k]):>14} {str(team[k]):>10}")
    out.append(f"{'turns by role':34} {single['turns by role']} | {team['turns by role']}")
    return "\n".join(out)


In [ ]:
%%writefile netops/trace.py
"""Printed reasoning traces: who said what, in order. Labs print them; tests read them."""
from __future__ import annotations


class Trace:
    def __init__(self, verbose: bool = True, width: int = 160):
        self.verbose, self.width, self.lines = verbose, width, []

    def __call__(self, who: str, text: str) -> None:
        self.lines.append((who, text))
        if self.verbose:
            for i, chunk in enumerate(text.splitlines() or [""]):
                print(f"{'[' + who + ']' if i == 0 else ' ' * (len(who) + 2):<20} {chunk[:self.width]}")
